## 1. Slowly Changing Dimensions (SCD) - comparison of announced Types

In dimensional modeling, a **slowly changing dimension** is a dimension whose attribute values change over time — slowly, not every transaction. Ralph Kimball defined six classic types (0–6); a seventh is commonly added in practice; an eighth (fact-date rows) was proposed by Oleg Ivanov in 2017, and this notebook develops its own concept — embedded history — as a ninth type (section 7). 

Each type answers the question: 
- when an attribute value changes?
- what do we do with the old value?

---

| Type | Author | Name | Strategy | Year announced | Old value preserved? | Full history? | New objects created |
|------|--------|------|---------|---------------|---------------------|---------------|--------------------|
| 0 | Ralph Kimball / Kimball Group | Retain / Original | Never change the value | 1996 (2008*) | ✅ (never overwritten) | ❌ | None — no schema or table changes |
| 1 | Ralph Kimball | Overwrite | Replace old with new | 1996 | ❌ | ❌ | None — same table, same columns, in-place updates |
| 2 | Ralph Kimball | Add New Row | Insert a new row, close out the old | 1996 | ✅ | ✅ | **New columns** added to the existing dimension: `valid_from`, `valid_to`, `is_current`, plus a **surrogate key** column (e.g. `product_sk`) |
| 3 | Ralph Kimball | Add New Column | Keep old + new in separate columns | 1996 | ✅ (one prior version) | Partial (current + previous) | **New columns** added to the existing dimension (e.g. `previous_category`) — one per tracked attribute |
| 4 | Ralph Kimball | Mini-Dimension | Rapidly changing attrs split into a separate dimension | 2002 | N/A (separate table) | ✅ (via mini-dim grouping) | **New table** (mini-dimension) for the rapidly-changing attributes, keyed by a surrogate key; base dimension gains a **new FK column** (e.g. `mini_sk`) pointing to it |
| 5 | Ralph Kimball | Mini-Dimension + Type 1 Outrigger | Type 4 + Type 1 ref in base dim | 2002 | Partial | ✅ (via mini-dim) | Everything from Type 4 **plus new outrigger columns** copied into the base dimension (e.g. `current_category`, `current_brand`) carrying the latest mini-dim values |
| 6 | Ralph Kimball | Hybrid (1 + 2) | Type 2 history + Type 1 overwrite for selected attrs | 2002 | Partial | ✅ | All Type 2 columns **plus a new current-value column** (e.g. `current_category`) stamped across every history row |
| 7 | Kimball Group | Dual Type (1 + 2) | Maintain both a Type 1 table and a Type 2 table | 2013 | Partial | ✅ | **Second physical table** — a Type 1 snapshot table alongside the Type 2 history table (two tables maintained in parallel) |
| 8 | Oleg Ivanov | Fact-Date Rows | One row per entity × fact date; single `event_date`; partitioned by `event_date` | 2017 | ✅ (value at each fact date) | ✅ (aligned to the facts) | No schema change — the *rows* multiply with the facts; special dates `1900-01-01` (initial) / `9999-12-31` (current) combine SCD 0 + SCD 1 in the same table |
| 9 | Bartosz Malec | Embedded History (∞) — concept | One row per entity; full history in an `ARRAY<STRUCT>` `_scd` column | 2026 (this notebook) | ✅ | ✅ | **New column** `_scd` (`ARRAY<STRUCT>`) added to the existing dimension — no new tables; each element holds one version with attributes, version number, and `valid_from`/`valid_to` |

---

*announced in 1996 with other concepts, but described in details in 2008.

## Detailed explanations

### Type 0 — Retain (Original Value)
Never modify the attribute — the value set at insert time is permanent, regardless of later changes in the source. Rarely used, but useful for immutable facts like a customer's original registration date or date of birth.

> **New objects:** None. The dimension table and its schema stay exactly as-is; no columns, tables, or lookup structures are added.
>
> **Announced:** Formally described by the Kimball Group in the third edition of *The Data Warehouse Toolkit* (2008), complementing the original three types.
> **Data Warehouse use case:** Preserving immutable attributes such as a customer's original registration date or date of birth that must never be modified.
> **Data Lakehouse use case:** Retaining original source-system landing timestamps in the Bronze layer for full audit traceability across the medallion architecture.

### Type 1 — Overwrite
Simply update the row in place. No history is kept — the old value is lost. This is the simplest and most common approach when only the current value matters (e.g., correcting a misspelled name).

> **New objects:** None. Same table, same columns — rows are simply overwritten in place.
>
> **Announced:** Introduced by Ralph Kimball in the first edition of *The Data Warehouse Toolkit* (1996) — one of the three original SCD types.
> **Data Warehouse use case:** Correcting a misspelled customer name or keeping the latest shipping address when only the current value matters.
> **Data Lakehouse use case:** Overwriting the latest record state in the Silver/Gold Delta tables via `MERGE` for downstream consumers that need only current values.

### Type 2 — Add New Row (Historical Tracking)
Insert a **new row** for the changed record and mark the previous row as expired. This preserves the full history of changes. Each row gets:
* **effective dates** (or timestamps): `valid_from` / `valid_to`
* a **current flag** (`is_current` = true/false)
* a **surrogate key** so every version has a unique primary key

This is the most widely used SCD type in data warehousing because it lets fact rows join to the dimension version that was active at the time of the fact.

> **New objects:** The **existing dimension table gains new columns** — at minimum a surrogate key (e.g. `product_sk`), `valid_from`, `valid_to`, and `is_current`. No additional tables are created.
>
> **Announced:** Introduced by Ralph Kimball in the first edition of *The Data Warehouse Toolkit* (1996) — one of the three original SCD types.
> **Data Warehouse use case:** Tracking full customer address history so that fact rows join to the dimension version that was active at the time of each transaction.
> **Data Lakehouse use case:** Maintaining full version history in Delta tables using Change Data Feed (CDF) with `MERGE` SCD Type 2 logic for point-in-time analytics and incremental CDC pipelines.

### Type 3 — Add New Column
Add a second column to hold the prior value (e.g., `previous_category` alongside `category`). Only one level of history is kept, so this is suitable when you only need to compare the current value to the immediately preceding one.

> **New objects:** **New columns** added to the existing dimension table — one `previous_*` column per tracked attribute (e.g. `previous_category`). No additional tables are created.
>
> **Announced:** Introduced by Ralph Kimball in the first edition of *The Data Warehouse Toolkit* (1996) — one of the three original SCD types.
> **Data Warehouse use case:** Comparing current vs. previous sales-territory assignment for an employee without the overhead of full history rows.
> **Data Lakehouse use case:** Lightweight one-level history tracking in a Silver Delta table when full SCD Type 2 is unnecessary but a before/after comparison is still required.

### Type 4 — Mini-Dimension
When certain attributes change **rapidly** (e.g., number of dependents, income band, credit score), they are split out of the main dimension into a separate *mini-dimension*. Each unique combination of those attributes becomes a row in the mini-dimension, keyed by a surrogate key. The base dimension references the current mini-dimension key (a Type 1 overwrite on that key). Fact rows then reference the mini-dimension surrogate key, capturing history without exploding the base dimension.

> **New objects:** A **new lookup table** (the mini-dimension) holding one row per unique attribute combination, keyed by its own surrogate key. The base dimension gains a **new foreign-key column** (e.g. `mini_sk`) referencing the mini-dimension.
>
> **Announced:** Introduced in the second edition of *The Data Warehouse Toolkit* (2002) to handle rapidly changing attributes.
> **Data Warehouse use case:** Managing fast-changing customer demographics (income band, credit score, dependents) without exploding the base customer dimension.
> **Data Lakehouse use case:** Splitting volatile attributes into a separate Delta table keyed by surrogate key, referenced from the main dimension to keep the base table compact and queryable.

### Type 5 — Mini-Dimension + Type 1 Outrigger
A Type 4 mini-dimension, plus the **base dimension also carries a Type 1 copy** of the key mini-dimension attributes as an outrigger (overwritten on change). This lets you group or filter by the *current* value of a rapidly changing attribute directly from the base dimension, while the mini-dimension preserves the full history.

> **New objects:** Everything from Type 4 (new mini-dimension table + `mini_sk` FK on the base) **plus new outrigger columns** on the base dimension (e.g. `current_category`, `current_brand`) that carry a Type 1 copy of the latest mini-dimension attribute values.
>
> **Announced:** Introduced in the second edition of *The Data Warehouse Toolkit* (2002) as an extension of the Type 4 mini-dimension pattern.
> **Data Warehouse use case:** Allowing BI dashboards to filter by the current income band directly from the base dimension while the mini-dimension preserves full historical accuracy.
> **Data Lakehouse use case:** Adding current-value outrigger columns in the Gold layer so BI tools can query volatile attributes without joining to the mini-dimension Delta table.

### Type 6 — Hybrid (Type 1 + Type 2)
Combines Type 2 (full history rows with effective dates) **and** Type 1 (overwrite) for a subset of columns. Typical implementation: the dimension has history rows (Type 2), but also maintains a **current-value column** that is always overwritten to the latest value (Type 1) across *all* rows. This lets you group all historical facts by the current value of an attribute, while still preserving the value as it was at each point in time.

> **New objects:** All Type 2 columns (`product_sk`, `valid_from`, `valid_to`, `is_current`) **plus a new current-value column** (e.g. `current_category`) added to the existing dimension table. No additional tables are created.
>
> **Announced:** Introduced in the second edition of *The Data Warehouse Toolkit* (2002) as a hybrid combining Types 1 and 2.
> **Data Warehouse use case:** Grouping all historical sales by the *current* product category while still preserving the category value as it was at each point in time.
> **Data Lakehouse use case:** Stamping a `current_*` column across every history row in a Delta table so a single Gold-layer query supports both current and point-in-time analytics.

### Type 7 — Dual Type 1 + Type 2
Maintain **two physical tables**: one Type 1 table (current state only, overwritten) and one Type 2 table (full history). Fact tables join to the Type 2 table for point-in-time accuracy, while the Type 1 table serves dashboards and queries that only need the latest snapshot. This trades extra storage for query simplicity.

> **New objects:** A **second physical table** — a Type 1 snapshot table alongside the Type 2 history table. The Type 2 table carries the SCD 2 columns (`product_sk`, `valid_from`, `valid_to`, `is_current`); the Type 1 table mirrors the original dimension schema. Two tables must be maintained and kept in sync.
>
> **Announced:** Type 7 was formally described in 2013 by the Kimball Group. It was included in the 3rd edition of The Data Warehouse Toolkit the same year.
> **Data Warehouse use case:** Serving a Type 1 current-state table for fast dashboards and a Type 2 history table for accurate point-in-time fact joins from the same source feed.
> **Data Lakehouse use case:** Maintaining a current-state Delta table (feeding BI/SQL warehouses) alongside a full-history Delta table (feeding incremental CDC and audit pipelines), both refreshed from the same batch or streaming source.

### Type 8 — Fact-Date Rows (Oleg Ivanov, 2017)
A row is added to the dimension for each combination of entity key and fact date; the `DATE_FROM`/`DATE_TO` construction of Type 2 is replaced with a single `EVENT_DATE`, and the table is partitioned by it — typically the same way as the fact table. A date-bounded query prunes to exactly the needed partitions (the optimizer applies the fact's date bound to the dimension's `event_date` by transitivity), and the equality join also allows precise cardinality estimates. The trade is disk space for performance: the article's example dimension grows from 21M (SCD 2) to 120M rows, while a typical one-month query reads 21× fewer rows.

> **New objects:** None in the schema — the change is in the *rows*: one per entity × fact date, plus "special" dates (`1900-01-01` for initial rows — SCD 0, `9999-12-31` for the last version — SCD 1) so several SCD types coexist in one table.
>
> **Announced:** Proposed by Oleg Ivanov in *"Introducing Slowly Changing Dimension Type 8"* (LinkedIn, 3 April 2017); the author notes the name and number are not yet established.
> **Data Warehouse use case:** High-cardinality, often-changed dimensions (e.g. a 1M-client dimension) where history is long but analyzed in small windows and facts are sparse per key — partition pruning beats full SCD 2 scans.
> **Data Lakehouse use case:** A Delta table partitioned (or liquid-clustered) by `event_date`, aligned to the fact table's date; `MERGE` applies new fact dates and restatements; equality joins on `sale_date = event_date` enable partition pruning.

### Type 9 — Embedded History (∞) — notebook concept
One row per entity stores **both current and historical data**: current attribute values live in scalar columns (no `is_current` flag), the full version history in an `_scd` `ARRAY<STRUCT>` column where each element holds one version with its attributes, version number, and `valid_from` / `valid_to`. The name "9" follows the next free number and, when rotated, reads as infinity (∞) — the schema of `_scd` is a free canvas.

> **New objects:** A **new column** `_scd` (`ARRAY<STRUCT>`) added to the existing dimension table. No additional tables are created — the full history lives inside the single row per entity.
>
> **Announced:** Bartosz Malec — this notebook's own concept (2026), developed and stress-tested in sections 7–14.
> **Data Warehouse use case:** Not traditionally applicable — designed specifically for Lakehouse-native patterns.
> **Data Lakehouse use case:** One row per entity in a Delta table with embedded history: current-value queries via column pruning, point-in-time analytics via `explode()` / higher-order functions — no separate history table, no multi-row joins.




### Parameters — catalog, schema, dimension and table names

Everything this notebook creates lives in one schema, set by the `catalog` / `schema` widgets. The `dimension_source` widget chooses **which dimension** is processed:

* `sample:product` (default) — the curated product dimension and its 60-event change feed that the narrative of this notebook uses,
* `<catalog>.<schema>.<table>` — **any** dimension table; set `business_key` too. Its change feed is simulated from the table's own rows (section 3).

Optional widgets: `dimension_name` (suffix of the table names), `type3_attr` (attribute tracked by Types 3 / 6) and `volatile_attrs` (mini-dimension attributes of Types 4 / 5). Left empty, they are derived from the data.

Every table is named `scd_<type>_<dimension>` (e.g. `scd_t2_dim_product`). No cell hard-codes a column name: the **business columns are read from the source schema**, the **technical SCD / ETL columns are listed once** in `TECH_COLS` (section 2), and every MERGE is generated from those lists by small SQL builders (`sql_cols`, `sql_set`, `sql_on`, …).

Run this cell first, before any cell that writes a table.

In [0]:
dbutils.widgets.text("catalog", "workspace")
dbutils.widgets.text("schema", "scd_concept")
# --- which dimension to process ------------------------------------------------------
#   "sample:product"              -> the curated product dimension + change feed of this notebook
#   "<catalog>.<schema>.<table>"  -> ANY dimension table; its change feed is simulated from its rows
dbutils.widgets.text("dimension_source", "sample:product")
dbutils.widgets.text("business_key", "")     # required for a table source (e.g. geo_code)
dbutils.widgets.text("dimension_name", "")   # optional label used in table names (default: from the source)
dbutils.widgets.text("type3_attr", "")       # optional: attribute tracked by Types 3 / 6 (default: most changed)
dbutils.widgets.text("volatile_attrs", "")   # optional: comma list for the Type 4 / 5 mini-dimension

catalog_name = dbutils.widgets.get("catalog")
schema_name  = dbutils.widgets.get("schema")
DIM_SOURCE   = dbutils.widgets.get("dimension_source").strip()
BUSINESS_KEY = dbutils.widgets.get("business_key").strip()
W_TYPE3_ATTR = dbutils.widgets.get("type3_attr").strip()
W_VOLATILE   = [c.strip() for c in dbutils.widgets.get("volatile_attrs").split(",") if c.strip()]

IS_SAMPLE = DIM_SOURCE.startswith("sample:")
DIM_NAME  = (dbutils.widgets.get("dimension_name").strip()
             or (DIM_SOURCE.split(":", 1)[1] if IS_SAMPLE
                 else DIM_SOURCE.split(".")[-1].removeprefix("dim_")))

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog_name}.{schema_name}")
spark.sql(f"USE CATALOG {catalog_name}")
spark.sql(f"USE SCHEMA {schema_name}")

def tbl(kind):
    """Full name of a table this notebook writes: scd_<kind>_<dimension>."""
    return f"{catalog_name}.{schema_name}.scd_{kind}_{DIM_NAME}"

TBL_T0           = tbl("t0_dim")
TBL_T1           = tbl("t1_dim")
TBL_T2           = tbl("t2_dim")
TBL_T3           = tbl("t3_dim")
TBL_T4_MINI      = tbl("t4_mini_dim")
TBL_T4_BASE      = tbl("t4_base_dim")
TBL_T5_MINI      = tbl("t5_mini_dim")
TBL_T5_BASE      = tbl("t5_base_dim")
TBL_T6           = tbl("t6_dim")
TBL_T6_CURRENT   = tbl("t6_current_values")
TBL_T7_CURRENT   = tbl("t7_dim_current")
TBL_T7_HISTORY   = tbl("t7_dim_history")
TBL_T8           = tbl("t8_dim")          # SCD Type 8 (Oleg Ivanov, 2017) — fact-date rows
TBL_T9           = tbl("t9_dim")          # SCD Type 9 (concept) — embedded _scd history
TBL_T9_EVOLUTION = tbl("t9_evolution")
TBL_T9_STRESS    = tbl("t9_stress")

ALL_TABLES = [TBL_T0, TBL_T1, TBL_T2, TBL_T3, TBL_T4_MINI, TBL_T4_BASE, TBL_T5_MINI, TBL_T5_BASE,
              TBL_T6, TBL_T6_CURRENT, TBL_T7_CURRENT, TBL_T7_HISTORY, TBL_T8, TBL_T9,
              TBL_T9_EVOLUTION, TBL_T9_STRESS]

print(f"Dimension source: {DIM_SOURCE}  ->  dimension name '{DIM_NAME}'")
print(f"All tables will be created in: {catalog_name}.{schema_name}")
print(f"Registered table names ({len(ALL_TABLES)}):")
for t in ALL_TABLES:
    print(f"  - {t}")

### IMPORTS — every install / import statement of the notebook, in one place

All libraries used anywhere below are imported here, right after the parameters cell, so no later cell carries its own import block.

In [0]:
import random
import re
import statistics
import time
from datetime import date, timedelta
from functools import reduce

from pyspark.sql import Row
from pyspark.sql import DataFrame
from pyspark.sql import functions as F
from pyspark.sql import types as T
from pyspark.sql.window import Window

print("Imports ready: random, re, statistics, time, datetime, functools.reduce | "
      "pyspark.sql: Row, DataFrame, F (functions), T (types), Window")


### Type 2 example (most common in practice)

Using the product sample above (`dim_source` / `dim_base`), if product 3 changes from category *Kitchen* to *Home*, a Type 2 table would look like:

| product_sk (surrogate) | product_id | product_name | category | valid_from | valid_to | is_current |
|------------------------|------------|--------------|----------|------------|----------|------------|
| 301 | 3 | Coffee Mug | Kitchen | 2024-01-01 | 2024-06-30 | false |
| 302 | 3 | Coffee Mug | Home | 2024-07-01 | 9999-12-31 | true |

The surrogate key `product_sk` is unique per version; the natural key `product_id` repeats. Fact rows reference `product_sk` so each fact is associated with the correct version of the product.

> **Reference:** [Slowly changing dimension — Wikipedia](https://en.wikipedia.org/wiki/Slowly_changing_dimension)

## 2. Sample Dimension table

In [0]:
# ===================================================================================
# 1. The dimension: either the curated sample below or ANY table (widget dimension_source)
# ===================================================================================
SAMPLE_DIMENSIONS = {
    "product": {
        "key": "product_id",
        "schema": "product_id long, product_name string, category string, brand string, color string",
        "rows": [
            (1, "Laptop",     "Electronics", "Acme",      "Silver"),
            (2, "Headphones", "Electronics", "Bose",      "Black"),
            (3, "Coffee Mug", "Kitchen",     "Ikea",      "White"),
            (4, "Notebook",   "Office",      "Moleskine", "Red"),
            (5, "Desk Lamp",  "Office",      "Ikea",      "Brass"),
        ],
        # optional hand-picked roles; anything left out is derived from the change feed
        "type3_attr": "category",
        "volatile_attrs": ["category", "brand", "color"],
        "outrigger_attrs": ["category", "brand"],
        "late_change": ("brand", "Bose"),            # Type 8 MERGE demo (section 8)
        "stress_new_col": ("stock_qty", "int"),      # stress test (section 14)
    },
}

if IS_SAMPLE:
    SAMPLE = SAMPLE_DIMENSIONS[DIM_SOURCE.split(":", 1)[1]]
    KEY = BUSINESS_KEY or SAMPLE["key"]
    dim_source = spark.createDataFrame(SAMPLE["rows"], SAMPLE["schema"])
else:
    SAMPLE = {}
    assert BUSINESS_KEY, "a table source needs the business_key widget (e.g. geo_code)"
    KEY = BUSINESS_KEY
    dim_source = spark.table(DIM_SOURCE)
assert KEY in dim_source.columns, f"business key '{KEY}' not found in {dim_source.columns}"

# ===================================================================================
# 2. Column model — business columns are READ from the source, technical ones are LISTED
# ===================================================================================
SK_COL   = f"{DIM_NAME}_sk"                                # surrogate key of Types 2 / 6 / 7
SCD_COLS = ["valid_from", "valid_to", "is_current"]        # versioning columns of Types 2 / 6 / 7
ETL_COLS = ["change_type", "change_date", "event_date"]    # change-feed / load metadata
TECH_COLS = {SK_COL, "mini_sk", "version", "_scd", *SCD_COLS, *ETL_COLS}   # extended below

def business_cols(df, exclude=()):
    """Attribute columns of df: everything except the key, the technical columns and 'exclude'."""
    skip = TECH_COLS | {KEY} | set(exclude)
    return [c for c in df.columns if c not in skip]

ATTRS = business_cols(dim_source)

def cols(*extra, attrs=None):
    """KEY + business attributes (+ the technical columns asked for), always in the same order."""
    return [KEY, *(ATTRS if attrs is None else attrs), *extra]

def col_type(name, df=None):
    """Spark SQL type of a column (from dim_source unless another DataFrame is given)."""
    return (dim_source if df is None else df).schema[name].dataType.simpleString()

# --- SQL builders: every generated statement gets its column lists from here ---------
BACKTICK = chr(96)

def q(c):
    """Quote a column name for SQL (backticks)."""
    return f"{BACKTICK}{c}{BACKTICK}"

def sql_cols(columns, prefix=""):
    return ", ".join(f"{prefix}{q(c)}" for c in columns)

def sql_list(*parts):
    """Comma-join SQL fragments, skipping empty ones (e.g. an empty attribute list)."""
    return ", ".join(p for p in parts if p)

def sql_set(columns, tgt="tgt", src="src"):
    return ",\n    ".join(f"{tgt}.{q(c)} = {src}.{q(c)}" for c in columns)

def sql_on(columns, a="tgt", b="src", null_safe=False):
    op = "<=>" if null_safe else "="
    return " AND ".join(f"{a}.{q(c)} {op} {b}.{q(c)}" for c in columns)

def sql_ddl(columns, df=None):
    return ",\n  ".join(f"{q(c)} {col_type(c, df)}" for c in columns)

def sql_lit(v):
    """SQL literal for a key value (strings quoted, numbers as they are)."""
    return "'" + str(v).replace("'", "''") + "'" if isinstance(v, str) else str(v)

def sql_struct(columns, prefix=""):
    """named_struct('c1', <prefix>c1, ...) — the payload of one _scd element."""
    return ", ".join(f"'{c}', {prefix}{q(c)}" for c in columns)

def null_safe_join(left, right, on, how="inner"):
    """Join on several columns treating NULL = NULL as a match (attribute combinations)."""
    cond = reduce(lambda a, b: a & b, [left[c].eqNullSafe(right[c]) for c in on])
    return left.join(right, cond, how).drop(*[right[c] for c in on])

# --- brand-new business keys (demo inserts, stress test) of the key's own type --------
KEY_TYPE = dim_source.schema[KEY].dataType
KEY_IS_INTEGRAL = isinstance(KEY_TYPE, T.IntegralType)
if KEY_IS_INTEGRAL:
    NEW_KEY_BASE = max(10_000_000, (dim_source.agg(F.max(KEY)).first()[0] or 0) + 1)

def new_key(n):
    """n-th brand-new key value (Python), never present in the source."""
    return NEW_KEY_BASE + n if KEY_IS_INTEGRAL else f"NEW-{n:08d}"

def new_key_col(n_col):
    """Same as new_key, as a Spark Column built from an integer Column."""
    if KEY_IS_INTEGRAL:
        return (F.lit(NEW_KEY_BASE) + n_col).cast(KEY_TYPE)
    return F.concat(F.lit("NEW-"), F.lpad(n_col.cast("string"), 8, "0"))

print(f"Business key : {KEY} ({KEY_TYPE.simpleString()})")
print(f"Attributes   : {ATTRS}")
print(f"Technical    : {sorted(TECH_COLS)}")
display(dim_source.orderBy(KEY))

## 3. Simulate incoming data batch

### Simulation of incoming data (change feed from the source system)

60 change events (10x the original batch): one feed, but carrying multiple change points spread over several months.

* insert → brand-new products (ids 6..22)
* update → full new snapshot of the record; the same `product_id` can appear many times with different `change_dates` (e.g. 1, 2, 3, 5, 6, 7, 8, 9, 10, 11, 20)
* delete → product removed from the source (only the key matters)

Every SCD type below consumes this same feed, each in its own way. **Note:** with multiple events per product, downstream logic typically has to pick the latest event per product (or replay events in date order) rather than blindly joining on `product_id`.


In [0]:
# Change feed rows: (key, <attributes in cols() order>, change_type, change_date)
SAMPLE_CHANGE_FEEDS = {
    "product": [
        # --- 2024-07-01: initial batch -----------------------------------
        # inserts: brand-new products
        (6,  "Mechanical Keyboard", "Electronics", "Logitech",      "Black",  "insert", "2024-07-01"),
        (7,  "Wall Clock",          "Home",        "Ikea",          "White",  "insert", "2024-07-01"),
        (8,  "Smart Watch",         "Electronics", "Garmin",        "Slate",  "insert", "2024-07-01"),
        (9,  "Yoga Mat",            "Sports",      "Decathlon",     "Blue",   "insert", "2024-07-01"),
        (10, "Blender",             "Kitchen",     "Philips",       "White",  "insert", "2024-07-01"),
        (11, "Office Chair",        "Office",      "Herman Miller", "Black",  "insert", "2024-07-01"),
        (12, "Desk Organizer",      "Office",      "Knoll",         "Grey",   "insert", "2024-07-01"),
        (13, "Water Bottle",        "Sports",      "CamelBak",      "Cyan",   "insert", "2024-07-01"),
        (14, "Air Purifier",        "Home",        "Dyson",         "Silver", "insert", "2024-07-01"),
        (15, "Gaming Mouse",        "Electronics", "Razer",         "Green",  "insert", "2024-07-01"),
        (16, "Standing Desk",       "Office",      "Varidesk",      "Oak",    "insert", "2024-07-01"),
        (17, "Curtains",            "Home",        "Zara Home",     "Beige",  "insert", "2024-07-01"),
        (18, "Coffee Grinder",      "Kitchen",     "Braun",         "Steel",  "insert", "2024-07-01"),
        (19, "Backpack",            "Travel",      "Samsonite",     "Red",    "insert", "2024-07-01"),
        (20, "Umbrella",            "Home",        "Totes",         "Navy",   "insert", "2024-07-01"),
        # updates: full new snapshot of existing products
        (1,  "Laptop",              "Computers",   "Acme",          "Silver", "update", "2024-07-01"),
        (2,  "Headphones",          "Electronics", "Sony",          "Black",  "update", "2024-07-01"),
        (3,  "Coffee Mug",          "Home",        "Ikea",          "White",  "update", "2024-07-01"),

        # --- August: first wave of follow-up changes ----------------------
        (5,  "Desk Lamp",           "Lighting",    "Ikea",          "Brass",  "update", "2024-08-05"),
        (2,  "Headphones",          "Electronics", "Sony",          "Beige",  "update", "2024-08-12"),  # 2nd change for product 2
        (6,  "Mechanical Keyboard", "Electronics", "Razer",         "Black",  "update", "2024-08-20"),
        (1,  "Laptop",              "Computers",   "Acme",          "Graphite", "update", "2024-08-25"),  # 2nd change for product 1
        (10, "Blender",             "Appliances",  "Philips",       "White",  "update", "2024-08-28"),
        (21, "Drone",               "Electronics", "DJI",           "Grey",   "insert", "2024-08-30"),  # late insert

        # --- September -----------------------------------------------------
        (12, None, None, None, None,                                          "delete", "2024-09-10"),  # product 12 removed
        (2,  "Headphones",          "Audio",       "Sony",          "Beige",  "update", "2024-09-15"),  # 3rd change for product 2
        (7,  "Wall Clock",          "Decor",       "Ikea",          "White",  "update", "2024-09-18"),
        (3,  "Coffee Mug",          "Home",        "Ikea",          "Ivory",  "update", "2024-09-22"),  # 2nd change for product 3
        (22, "Bluetooth Speaker",   "Electronics", "Bose",          "Black",  "insert", "2024-09-25"),
        (5,  "Desk Lamp",           "Lighting",    "Philips",       "Brass",  "update", "2024-09-28"),  # 2nd change for product 5
        (1,  "Laptop Pro",          "Computers",   "Acme",          "Graphite", "update", "2024-09-30"),  # renamed
        (8,  "Smart Watch",         "Electronics", "Garmin",        "Black",  "update", "2024-09-30"),

        # --- October -------------------------------------------------------
        (10, "Blender",             "Appliances",  "Philips",       "Cream",  "update", "2024-10-05"),
        (4,  "Notebook",            "Office",      "Moleskine",     "Black",  "update", "2024-10-08"),
        (20, "Umbrella",            "Home",        "Totes",         "Green",  "update", "2024-10-12"),
        (17, "Curtains",            "Home",        "Zara Home",     "White",  "update", "2024-10-15"),
        (9,  "Yoga Mat",            "Sports",      "Decathlon",     "Purple", "update", "2024-10-18"),
        (2,  "Headphones",          "Audio",       "Sony",          "Navy",   "update", "2024-10-22"),  # 4th change for product 2
        (13, "Water Bottle",        "Sports",      "CamelBak",      "Charcoal", "update", "2024-10-25"),
        (14, "Air Purifier",        "Appliances",  "Dyson",         "Silver", "update", "2024-10-28"),
        (4,  None, None, None, None,                                          "delete", "2024-10-31"),  # updated earlier in Oct, removed at month end

        # --- November ------------------------------------------------------
        (6,  "Mechanical Keyboard", "Electronics", "Razer",         "White",  "update", "2024-11-04"),  # 2nd change for product 6
        (11, "Office Chair",        "Office",      "Herman Miller", "Grey",   "update", "2024-11-08"),
        (1,  "Laptop Pro",          "Computers",   "Acer",          "Graphite", "update", "2024-11-12"),  # 4th change for product 1
        (16, "Standing Desk",       "Office",      "Varidesk",      "Walnut", "update", "2024-11-15"),
        (10, "Blender",             "Appliances",  "Vitamix",       "Cream",  "update", "2024-11-18"),  # 3rd change for product 10
        (14, None, None, None, None,                                          "delete", "2024-11-20"),  # product 14 removed
        (19, "Backpack",            "Travel",      "Samsonite",     "Black",  "update", "2024-11-22"),
        (5,  "Desk Lamp",           "Lighting",    "Philips",       "Matte Black", "update", "2024-11-25"),  # 3rd change for product 5
        (8,  "Smart Watch",         "Electronics", "Apple",         "Black",  "update", "2024-11-28"),  # 2nd change for product 8

        # --- December ------------------------------------------------------
        (2,  "Headphones",          "Audio",       "Sennheiser",    "Navy",   "update", "2024-12-02"),  # 5th change for product 2
        (7,  "Wall Clock",          "Decor",       "Ikea",          "Oak",    "update", "2024-12-05"),  # 2nd change for product 7
        (15, "Gaming Mouse",        "Electronics", "Razer",         "Pink",   "update", "2024-12-08"),
        (22, "Bluetooth Speaker",   "Electronics", "Sonos",         "Black",  "update", "2024-12-10"),
        (9,  "Yoga Mat",            "Sports",      "Lululemon",     "Purple", "update", "2024-12-12"),  # 2nd change for product 9
        (3,  "Coffee Mug",          "Home",        "Bodum",         "Ivory",  "update", "2024-12-15"),  # 3rd change for product 3
        (18, "Coffee Grinder",      "Kitchen",     "Baratza",       "Steel",  "update", "2024-12-18"),
        (11, "Office Chair",        "Office",      "Steelcase",     "Grey",   "update", "2024-12-20"),  # 2nd change for product 11
        (17, None, None, None, None,                                          "delete", "2024-12-22"),  # product 17 removed
        (20, "Umbrella",            "Home",        "Windbrella",    "Green",  "update", "2024-12-28"),  # 2nd change for product 20
    ],
}

# --- generic simulator for ANY dimension table (used when no curated feed exists) -----
FEED_SEED = 42
MAX_SIMULATED_ROWS = 2000   # the SCD demos only need a sample of a large dimension

def simulate_change_feed(rows, seed=FEED_SEED):
    """Split the rows into an initial snapshot and a 2024 change feed.

    * the last quarter of the rows (by key) is held out and arrives as INSERTs in Jul-Sep,
    * every entity gets 0-4 UPDATEs (1-2 attributes take a value seen in another row),
    * about 1 in 8 entities is DELETEd after its last update.
    Rows are tuples in cols() order; the result is deterministic for a given seed."""
    rnd = random.Random(seed)
    rows = sorted(rows, key=lambda r: str(r[0]))
    pools = [sorted({r[i] for r in rows if r[i] is not None}, key=str) or [None]
             for i in range(1, len(ATTRS) + 1)]
    n_hold = max(1, len(rows) // 4)
    initial, held_out = rows[:-n_hold], rows[-n_hold:]
    days = [date(2024, 7, 1) + timedelta(days=d) for d in range(181)]     # 2024-07-01 .. 2024-12-28
    events = []

    def lifecycle(row, first_day_idx, insert):
        state = list(row)
        day = first_day_idx
        if insert:
            events.append((*state, "insert", days[day].isoformat()))
        for _ in range(rnd.choice([0, 1, 1, 2, 2, 3, 4])):
            day = rnd.randint(day + 1, min(day + 60, len(days) - 2)) if day + 1 < len(days) - 1 else None
            if day is None:
                break
            for i in rnd.sample(range(1, len(state)), k=min(rnd.choice([1, 1, 2]), len(state) - 1)):
                choices = [v for v in pools[i - 1] if v != state[i]] or pools[i - 1]
                state[i] = rnd.choice(choices)
            events.append((*state, "update", days[day].isoformat()))
        if rnd.random() < 0.125 and day + 1 < len(days):
            del_day = rnd.randint(day + 1, len(days) - 1)
            events.append((state[0], *[None] * len(ATTRS), "delete", days[del_day].isoformat()))

    for r in initial:
        lifecycle(r, rnd.randint(0, 30), insert=False)
    for r in held_out:
        lifecycle(r, rnd.randint(0, 91), insert=True)
    return initial, sorted(events, key=lambda e: (e[-1], str(e[0])))

NULLABLE = lambda schema: T.StructType([T.StructField(f.name, f.dataType, True) for f in schema])
BASE_SCHEMA = NULLABLE(dim_source.select(*cols()).schema)

if IS_SAMPLE and DIM_NAME in SAMPLE_CHANGE_FEEDS:
    initial_rows, change_events = SAMPLE["rows"], SAMPLE_CHANGE_FEEDS[DIM_NAME]
    print(f"Curated change feed: {len(change_events)} events")
else:
    source_rows = [tuple(r) for r in dim_source.select(*cols()).orderBy(KEY).limit(MAX_SIMULATED_ROWS).collect()]
    initial_rows, change_events = simulate_change_feed(source_rows)
    print(f"Simulated change feed (seed {FEED_SEED}): {len(change_events)} events over "
          f"{len(source_rows)} source rows — {len(source_rows) - len(initial_rows)} held out as later inserts")

# the dimension as of 2024-01-01 (for a table source: without the held-out rows)
dim_base = spark.createDataFrame(initial_rows, BASE_SCHEMA)

changes = (
    spark.createDataFrame(
        change_events,
        T.StructType(BASE_SCHEMA.fields + [T.StructField("change_type", T.StringType()),
                                           T.StructField("change_date", T.StringType())]))
    .withColumn("change_date", F.to_date("change_date"))
    .orderBy("change_date", KEY)
)
display(changes)

events_with_initial = (
    dim_base.select(*cols())
    .withColumn("change_type", F.lit("initial"))
    .withColumn("change_date", F.to_date(F.lit("2024-01-01")))
    .unionByName(changes.select(*cols("change_type", "change_date")))
)

latest_events = (
    events_with_initial
    .withColumn("rn", F.row_number().over(Window.partitionBy(KEY).orderBy(F.col("change_date").desc())))
    .filter(F.col("rn") == 1)
    .drop("rn")
)

# --- attribute roles for the types that treat attributes differently (3, 4, 5, 6) ----
# Default: rank the attributes by how often they really change in the feed.
w_k = Window.partitionBy(KEY).orderBy("change_date")
CHANGE_COUNTS = (
    events_with_initial.filter(F.col("change_type") != "delete")
    .select(*[((F.row_number().over(w_k) > 1) & ~F.col(a).eqNullSafe(F.lag(a).over(w_k))).cast("int").alias(a)
              for a in ATTRS])
    .agg(*[F.sum(a).alias(a) for a in ATTRS])
    .first().asDict()
)
RANKED_ATTRS = sorted(ATTRS, key=lambda a: -(CHANGE_COUNTS[a] or 0))

TYPE3_ATTR      = W_TYPE3_ATTR or SAMPLE.get("type3_attr") or RANKED_ATTRS[0]
VOLATILE_ATTRS  = W_VOLATILE or SAMPLE.get("volatile_attrs") or \
                  [a for a in ATTRS if a in RANKED_ATTRS[:max(1, len(ATTRS) // 2)]]
STABLE_ATTRS    = [a for a in ATTRS if a not in VOLATILE_ATTRS]
OUTRIGGER_ATTRS = SAMPLE.get("outrigger_attrs") or VOLATILE_ATTRS[:2]
TYPE6_ATTRS     = [TYPE3_ATTR]
assert {TYPE3_ATTR, *VOLATILE_ATTRS, *OUTRIGGER_ATTRS} <= set(ATTRS), "role attributes must be business columns"

PREV_COL = f"previous_{TYPE3_ATTR}"           # Type 3

def current_col(a):                           # Types 5 / 6
    return f"current_{a}"

TECH_COLS.update({PREV_COL, *[current_col(a) for a in OUTRIGGER_ATTRS + TYPE6_ATTRS]})

print("Changes per attribute :", CHANGE_COUNTS)
print(f"Type 3 / 6 attribute  : {TYPE3_ATTR}  (-> {PREV_COL}, {current_col(TYPE3_ATTR)})")
print(f"Type 4 / 5 mini-dim   : {VOLATILE_ATTRS}   stable: {STABLE_ATTRS}   outrigger: {OUTRIGGER_ATTRS}")

## 4. Behaviour of each SCD Type

### Type 0 — retain (original value)

* inserts → applied (new rows added once, then never touched again)
* updates → ignored (original value kept forever)
* deletes → ignored (row stays; Type 0 never changes anything)


In [0]:
inserts = changes.filter(F.col("change_type") == "insert")

dim_t0 = dim_base.unionByName(
    inserts.select(*cols())
)

display(dim_t0.orderBy(KEY))

### Type 1 — overwrite

* inserts → added as new rows
* updates → existing row overwritten in place (old value lost)
* deletes → row removed


In [0]:
dim_t1 = (
    latest_events
    .filter(F.col("change_type") != "delete")
    .select(*cols())
)

display(dim_t1.orderBy(KEY))

### Type 2 — add new row (full history)

* inserts → new current rows
* updates → old version closed (`valid_to` = `change_date` - 1, `is_current` = false) + new version inserted with a fresh surrogate key
* deletes → old version closed, no replacement row


In [0]:
w_t2 = Window.partitionBy(KEY).orderBy("change_date")

t2_versions = (
    events_with_initial
    .withColumn("next_date", F.lead("change_date").over(w_t2))
    .filter(F.col("change_type") != "delete")
    .withColumn("valid_from", F.col("change_date"))
    .withColumn("valid_to", F.coalesce(F.date_sub(F.col("next_date"), 1), F.to_date(F.lit("9999-12-31"))))
    .withColumn("is_current", F.col("next_date").isNull())
)

dim_t2_final = (
    t2_versions
    # deterministic per version and computed without a global (single-partition) window:
    # the same version always gets the same key, on every rerun
    .withColumn(SK_COL, F.xxhash64(KEY, "valid_from"))
    .select(*cols(SK_COL, *SCD_COLS))
)

display(dim_t2_final.orderBy(KEY, "valid_from"))
display(
    dim_t2_final.filter(F.col("is_current"))
    .select(*cols(SK_COL, "valid_from", "valid_to"))      # every column except is_current
    .orderBy(KEY)
)

### Type 3 — add new column (current + previous)

* inserts → added as new rows (no previous value yet)
* updates → old value moved to `previous_category`, new value becomes current
* deletes → row removed


In [0]:
current_t3 = (
    latest_events
    .filter(F.col("change_type") != "delete")
    .select(*cols())
)

previous_t3 = (
    events_with_initial
    .filter(F.col("change_type") != "delete")
    .join(current_t3.select(KEY, F.col(TYPE3_ATTR).alias("_current_value")), KEY)
    .filter(~F.col(TYPE3_ATTR).eqNullSafe(F.col("_current_value")))
    .groupBy(KEY)
    .agg(F.max_by(TYPE3_ATTR, "change_date").alias(PREV_COL))
)

dim_t3 = current_t3.join(previous_t3, KEY, "left")

display(dim_t3.orderBy(KEY))

### Type 4 — mini-dimension

Rapidly changing attributes (category, brand, color) are split out of the base dimension into a mini-dimension: one row per unique attribute combination. The base dimension keeps only stable attributes plus a reference (`mini_sk`) to the current group — that reference is overwritten Type 1-style on change. History lives in the fact table's `mini_sk` references, not in the dimension itself.

* inserts → new group(s) in the mini-dim + new base row pointing at them
* updates → base row re-pointed to the (possibly new) matching group
* deletes → base row removed; groups stay (historical facts still use them)


In [0]:
all_combos = (
    events_with_initial
    .filter(F.col("change_type") != "delete")
    .select(*VOLATILE_ATTRS)
    .distinct()
)

mini_dim_t4 = all_combos.withColumn(
    "mini_sk", (F.row_number().over(Window.orderBy(*VOLATILE_ATTRS)) + F.lit(1000)).cast("long")
)

# stable attributes keep the value of the entity's first version
w_first = Window.partitionBy(KEY).orderBy("change_date")
stable_t4 = (
    events_with_initial
    .filter(F.col("change_type") != "delete")
    .withColumn("rn", F.row_number().over(w_first))
    .filter(F.col("rn") == 1)
    .select(KEY, *STABLE_ATTRS)
)

base_dim_t4_final = (
    null_safe_join(
        latest_events.filter(F.col("change_type") != "delete").select(KEY, *VOLATILE_ATTRS),
        mini_dim_t4, VOLATILE_ATTRS)
    .join(stable_t4, KEY)
    .select(KEY, *STABLE_ATTRS, "mini_sk")
)

print("Mini-dimension (one row per unique attribute combination):")
display(mini_dim_t4.orderBy("mini_sk"))
print("Base dimension (stable attributes + current group reference):")
display(base_dim_t4_final.orderBy(KEY))

### Type 5 — mini-dimension + Type 1 outrigger

Type 4 (cell above) plus the base dimension carries a Type 1 copy of the key mini-dimension attributes as an "outrigger" — overwritten on change. This lets you group/filter by the current value of a rapidly changing attribute directly from the base dimension, while the mini-dimension preserves the full history via `mini_sk` references.

Requires the Type 4 cell above to have been executed (uses `base_dim_t4_final` and `mini_dim_t4`).


### Outrigger

The outrigger holds the current group attributes copied into the base dimension. In a real warehouse these columns are stored and overwritten (Type 1); here they are derived from the current `mini_sk` reference, which gives the same result.


In [0]:
base_dim_t5 = (
    base_dim_t4_final
    .join(mini_dim_t4, "mini_sk")
    .select(KEY, *STABLE_ATTRS, "mini_sk",
            *[F.col(a).alias(current_col(a)) for a in OUTRIGGER_ATTRS])
)

display(base_dim_t5.orderBy(KEY))
# Note: entities whose current combination of the mini-dimension attributes is the same
# share one mini_sk — that sharing is the whole point of the mini-dimension.

### Type 6 — hybrid (Type 2 history + Type 1 current-value column)

Full history exactly like Type 2 (reusing `dim_t2_final` from the Type 2 cell above), plus a `current_<attr>` column (`current_category` for the product sample) that carries the latest value on every historical row — so old facts can also be grouped by the current attribute value, while the original columns still show the value as it was at each point in time.

Requires the Type 2 cell above to have been executed (uses `dim_t2_final`).


In [0]:
# --- Step 1: latest value(s) per entity (from the current versions) ---
current_values = (
    dim_t2_final
    .filter(F.col("is_current"))
    .select(KEY, *[F.col(a).alias(current_col(a)) for a in TYPE6_ATTRS])
)

# --- Step 2: stamp them onto every row, old versions included ---
dim_t6_final = dim_t2_final.join(current_values, KEY, "left")

display(dim_t6_final.orderBy(KEY, "valid_from"))
# Note: an entity whose TYPE6 attribute changed shows the old value on its early versions and
# the new one later, but the same current_<attr> on ALL of its rows. Deleted entities have no
# current version, so their historical rows keep current_<attr> = null.

### Type 7 — dual Type 1 + Type 2 (two physical tables)

Both tables are maintained in parallel from the same change batch:

* `dim_t1`       (Type 1 cell) → current snapshot for dashboards
* `dim_t2_final` (Type 2 cell) → full history for point-in-time joins

* inserts → added to both tables
* updates → overwritten in the Type 1 table; new version row in the Type 2 table
* deletes → removed from the Type 1 table; last version closed in the Type 2 table and kept as history

Requires the Type 1 and Type 2 cells above to have been executed (uses `dim_t1` and `dim_t2_final`).


In [0]:
print("Type 1 table — current snapshot (dashboards / latest value):")
display(dim_t1.orderBy(KEY))
print("Type 2 table — full history (point-in-time fact joins):")
display(dim_t2_final.orderBy(KEY, "valid_from"))

### Type 8 — fact-date rows (Oleg Ivanov, 2017)

A row is added to the dimension for each combination of entity key and fact date. A single `event_date` replaces the `valid_from` / `valid_to` pair, and the table is partitioned by `event_date` — the same way the fact table is partitioned by its date.

* inserts → an initial row at the special date `1900-01-01` (SCD 0 style) and a current-version row at `9999-12-31` (SCD 1 style); further rows appear automatically as facts with new dates arrive
* updates → the `9999-12-31` row is overwritten in place (SCD 1); rows for future fact dates carry the new values; existing fact-date rows keep the values that were true at their date (unless deliberately restated)
* deletes → not covered by the article; practical handling: the product stops getting new fact dates and its `9999-12-31` row is removed or closed, while the fact-date rows keep the historical truth

### Type 9 — embedded history (single row with `_scd` array) — notebook concept

* inserts → new row with `_scd` = [version 0]
* updates → current columns overwritten in place (Type 1 style) and a new version appended to `_scd` (version 1, 2, …)
* deletes → last version in `_scd` closed (`valid_to` < 9999-12-31), row kept — no flag column; the closed last version is what marks the delete

## 5. Review of currently used SCD Types

### The questions SCD Types 0–9 answer

Each SCD type is really an answer to a specific business question about **how attribute values change over time**.
Pick the type by asking: *what do I need to know about this attribute's history, and how far back?*

| Type | Author | The question it answers |
|------|--------|------------------------|
| **Type 0 — Retain original** | Ralph Kimball / Kimball Group | *"What was the value when the record was first loaded, and should it ever change?"* <br>The attribute is treated as permanent — original value kept forever, no updates applied (e.g. date of birth, original hire date, national ID). |
| **Type 1 — Overwrite** | Ralph Kimball | *"What is the **current** value?"* <br>Only the latest state matters; history is discarded. Wrong-value corrections and reporting that never needs point-in-time accuracy. |
| **Type 2 — Add new row** | Ralph Kimball | *"What was the value **at a specific point in time**, so I can join facts to the attributes that were in effect when the event happened?"* <br>Full history: every change is a new versioned row with `valid_from` / `valid_to` — point-in-time joins, audits, restating history. |
| **Type 3 — Add new column** | Ralph Kimball | *"What is the current value, and what was it **just before** that?"* <br>One step of history only (current + previous). *"What category is this product in now, and what category did we move it from?"* |
| **Type 4 — Mini-dimension** | Ralph Kimball | *"Which combination of rapidly changing attributes was in effect over time — without exploding the base dimension with thousands of rows per entity?"* <br>Groups of attribute values become their own dimension; history lives in the fact table's `mini_sk` references. |
| **Type 5 — Mini-dim + Type 1 outrigger** | Ralph Kimball | *"What is the **current** group of rapid attributes directly from the base dimension, while still being able to reconstruct **past** groupings?"* <br>Type 4 history plus a Type 1 copy of the key current attributes, so dashboards can filter by current value without a join through the facts. |
| **Type 6 — Hybrid (1+2)** | Ralph Kimball | *"What was the value **at the time of the fact**, and what is the **latest** value — on the same row?"* <br>Full Type 2 history plus an overwritten current-value column. *"Sales per current category, even though the fact was recorded under the old one."* |
| **Type 7 — Dual (1+2 tables)** | Kimball Group | *"Can I have both a fast, simple current snapshot **and** complete history — without one compromising the other?"* <br>Two physical tables maintained in parallel: Type 1 for dashboards/latest values, Type 2 for point-in-time fact joins. |
| **Type 8 — Fact-date rows (Oleg Ivanov, 2017)** | Oleg Ivanov | *"What did the entity look like exactly on the date of **this** fact — and can I read it cheaply?"* <br>One row per entity × fact date with a single `event_date`; partition pruning turns the as-of read into an equality join instead of a range scan. |
| **Type 9 — Embedded history (notebook concept)** | Bartosz Malec | *"Can one single row carry the current values **and** the entire history — natively in the Lakehouse?"* <br>Current values in regular columns, the full version timeline in an `_scd` `ARRAY<STRUCT>`. |

**The pattern:** as you go from Type 0 → 7, the question grows from
*"never changes"* → *"what is current?"* → *"what was it then?"* → *"then and now, together"* →
*"many attributes changing fast — group them"* → *"fast current reporting plus full history"*.
Each step trades simplicity for richer time-awareness (or, as in Types 6–7, pays with extra structure/tables to get both at once).

Type 8 (Oleg Ivanov, 2017) answers: *"What did the entity look like exactly on the date of this fact — at partition-pruned read cost?"* — trading disk space for query speed. Type 9, this notebook's own concept (section 7), answers: *"Can one single row carry the current values **and** the entire history — natively in the Lakehouse?"*

### What problems can't SCD Types 0–8 solve?

Each type answers a time-awareness question, but every one of them forces a **trade-off** — none gives you *current values* and *complete history* together in a **single row**, natively, in the Lakehouse. The announced **SCD Type 8** (Oleg Ivanov, 2017) attacks the same problem from a different angle: it trades disk space for query speed by storing one row per entity **per fact date** (see section 6) — but even there current values and complete history do not live in a single row. The remaining gaps motivate this notebook's own concept, **SCD Type 9**:

**1. Current snapshot + full history always requires duplication.**
Type 1 (overwrite) loses history; Type 2 keeps it only by multiplying rows. To get both you must either maintain **two physical tables** (Type 7 — double storage, double ETL, risk of drift between them) or bolt an outrigger onto a mini-dimension (Type 5 — still multiple objects and joins).

**2. Row explosion for fast-changing dimensions.**
Type 2 adds a new row per change; for rapidly changing attributes this bloats the base dimension to thousands of rows per entity. Type 4 fixes it only by moving history into the fact table (`mini_sk`), which means point-in-time analysis is only possible through fact joins — never directly from the dimension.

**3. History is only one step deep — or needs an explosion.**
Type 3 stores just *previous*; anything deeper is impossible. Type 2/6 history requires **exploding rows and re-joining** to reconstruct a timeline — there is no way to observe an entity's evolution over time directly on one row.

**4. Every version must fit one rigid schema.**
All Types 0–7 assume a single fixed column set for the whole table. A new attribute forces a full schema change and backfilling nulls for historical rows — there is no *elasticity*, i.e. no way for each version of a record to carry the schema that was actual at that time.

**5. "Current" always costs something.**
Type 2 needs an `is_current` flag + filter (every query touches the whole versioned table); Type 6 adds an overwritten column that must be recomputed on every historical row; Type 5 needs an outrigger join. Dashboards can never read plain, pruned, single-row-per-entity current columns while history sits elsewhere in the same object.

**6. No Lakehouse-native single-object design.**
Types 2–7 model history through *separate rows, separate tables, or separate dimensions* — structures inherited from traditional warehouses. None exploits the Lakehouse's ability to embed history in one row via `ARRAY<STRUCT>`, with column pruning for current-value reporting and higher-order functions / `explode()` for history.

**The gap SCD Type 9 fills:**
one row per entity — current values in regular columns (Type 1-style overwrite) **and** the complete version history embedded in a `_scd` array (version, `valid_from`/`valid_to`, values actual at that time) — current reporting never touches `_scd` (column pruning), history is one `explode()` or higher-order function away, deletes are marked by a closed last version instead of a flag, and the `_scd` schema is a free canvas ("infinity") for whatever a business phenomenon requires.

## 6. Concept of SCD Type 8 — one row per entity × fact date (Oleg Ivanov, 2017)

> **Source article:** [Introducing Slowly Changing Dimension Type 8](https://www.linkedin.com/pulse/introducing-slowly-changing-dimension-type-8-oleg-ivanov/) — Oleg Ivanov, 3 April 2017. The author himself notes that the name and number are not yet established; this notebook follows his criteria.

### Criteria from the article

SCD Type 8 targets dimensions that have **high cardinality** and are **changed relatively often**. It differs from SCD Type 2 in two major aspects:

1. **`DATE_FROM` / `DATE_TO` construction is replaced with just one `EVENT_DATE`.**
2. **A row is added into the dimension table for each combination of entity key and fact date** (e.g. `CLIENT_ID` × `SALE_DATE`).

One row per entity × fact date sounds absurd — until **partitioning** is considered: the table is partitioned by `EVENT_DATE`, in the same manner as the fact table. The article's worked example: 1,000,000 clients × 10 years of history × 2 changes per year gives an SCD 2 dimension of

`1 000 000 * (2 * 10 + 1) = 21 000 000` rows,

all of which must be read on each access (SCD 2 tables are not convenient for date partitioning). The SCD 8 dimension instead holds

`1 000 000 * 10 * 12 = 120 000 000` rows,

but a query analyzing one month reads only that month's partition — **1 000 000 rows, 21× less**. The optimizer applies the fact table's date boundary to the dimension's `event_date` via the **transitivity rule**, enabling partition pruning. The classic IT trade: **sacrifice disk space to gain performance** (here 6× space for a 21× smaller read).

**When SCD 8 is more useful** (per the article):

* a **long period of history** has to be stored,
* a **relatively small period of history is analyzed** each time,
* the **dimension is changed often**,
* **facts happen relatively rarely** for the dimension key.

**Additional bonuses listed in the article:**

* SCD 8 **combines easily with SCD 0 and/or SCD 1 in the same table** using "special" dates — e.g. `1900-01-01` for initial rows and `9999-12-31` for the last version — with no performance penalty, because SCD 0/1 data resides in separate partitions.
* An optimizer can calculate **cardinality very precisely** for the equality predicates, while SCD 2's `<DATE> BETWEEN DATE_FROM AND DATE_TO` forces guesses.
* With an In-Memory feature, the **last or most-often-used dates can be cached**, as such features work on partition level.
* **Partition-wise join** can be used to join fact and dimension.

What the article does **not** cover, and this notebook handles explicitly: **deletes** (stop generating fact-date rows after the delete; no `9999-12-31` row for deleted entities), and **restatements** (correcting the attributes of an already-stored fact-date row — done via the `WHEN MATCHED` branch of the MERGE in section 8).

### Type 8 — applied to the dimension (product sample) (per the article's criteria)

* one dimension row per **combination of `business key` and sale date** from the simulated sales fact table, holding the attribute values **as they were on that date**
* a **single `event_date`** column — no `valid_from` / `valid_to`, no `is_current` flag
* **special dates** combine SCD 0 / SCD 1 in the same table: `1900-01-01` holds each product's initial state (SCD 0), `9999-12-31` holds the latest state, overwritten in place (SCD 1)
* partitioned (in the Delta table of section 8) **by `event_date`**, so month-bounded queries prune to one partition
* **as-of access is an equality join** — `fact.sale_date = dim.event_date` — instead of a date-range `BETWEEN`, which is what makes the precise cardinality estimates and partition-wise joins of the article possible


In [0]:
# --- SCD Type 8 (Oleg Ivanov, 2017): one row per (business key, sale_date) ---
# Article criteria implemented here:
#   * a single EVENT_DATE instead of DATE_FROM / DATE_TO (no is_current flag)
#   * a dimension row for each combination of entity key and fact date
#   * partitioned by EVENT_DATE (materialised in section 8)
#   * special dates: 1900-01-01 = initial row (SCD 0), 9999-12-31 = last version (SCD 1)

# --- Step 1: entity states over time (initial snapshot + non-delete events) ---
states_t8 = (
    dim_base.select(*cols(), F.to_date(F.lit("2024-01-01")).alias("state_from"))
    .unionByName(
        changes.filter(F.col("change_type") != "delete")
        .select(*cols(), F.col("change_date").alias("state_from"))
    )
)

# deleted entities: facts stop at the delete date, and they get no 9999-12-31 row
deleted_ids_t8 = changes.filter(F.col("change_type") == "delete").select(KEY).distinct()

# only DELETE events end an entity's sales; an update does not stop it selling
last_event_t8 = (
    changes.filter(F.col("change_type") == "delete")
    .groupBy(KEY)
    .agg(F.max("change_date").alias("last_event"))
)

# --- Step 2: simulated sales fact — one sale per entity per month ---------------
# The article: "a row ... for each combination of CLIENT_ID and SALE_DATE".
# Sales span each entity's life: from its first appearance to its delete (or year end).
month_bounds_t8 = (
    states_t8.groupBy(KEY).agg(F.min("state_from").alias("first_seen"))
    .join(last_event_t8, KEY, "left")
    .withColumn("last_active", F.coalesce(F.date_sub(F.col("last_event"), 1), F.to_date(F.lit("2024-12-01"))))
    .withColumn("first_month", F.date_trunc("month", F.col("first_seen")).cast("date"))
    .withColumn("last_month", F.least(F.date_trunc("month", F.col("last_active")).cast("date"),
                                      F.to_date(F.lit("2024-12-01"))))
    .withColumn("sale_dates", F.expr("sequence(first_month, last_month, INTERVAL 1 MONTH)"))
)
sales = month_bounds_t8.select(KEY, F.explode("sale_dates").alias("sale_date"))

# same-day second sales for the two lowest keys in October: the dimension still keeps
# ONE row per (key, sale_date) combination, not one per fact row
DUP_KEYS = [r[0] for r in dim_base.select(KEY).orderBy(KEY).limit(2).collect()]
sales = (
    sales
    .unionByName(sales.filter(F.col(KEY).isin(DUP_KEYS) & (F.month(F.col("sale_date")) == 10)))
    .cache()
)
print(f"Sales fact rows (incl. same-day duplicates): {sales.count()}")

# --- Step 3: one dimension row per (key, sale_date) -----------------------------
# Attribute values as they were ON that date: the latest state_from <= sale_date.
w_asof_t8 = Window.partitionBy(KEY, "sale_date").orderBy(F.col("state_from").desc())
fact_rows_t8 = (
    sales.select(KEY, "sale_date").distinct()
    .join(states_t8, KEY)
    .filter(F.col("sale_date") >= F.col("state_from"))
    .withColumn("rn", F.row_number().over(w_asof_t8))
    .filter(F.col("rn") == 1)
    .select(*cols(), F.col("sale_date").alias("event_date"))
)

# --- Step 4: special dates — SCD 0 (initial) and SCD 1 (current) rows -----------
w_first_t8 = Window.partitionBy(KEY).orderBy("state_from")
initial_rows_t8 = (
    states_t8.withColumn("rn", F.row_number().over(w_first_t8)).filter(F.col("rn") == 1)
    .select(*cols(), F.to_date(F.lit("1900-01-01")).alias("event_date"))
)
w_last_t8 = Window.partitionBy(KEY).orderBy(F.col("state_from").desc())
current_rows_t8 = (
    states_t8.join(deleted_ids_t8, KEY, "left_anti")
    .withColumn("rn", F.row_number().over(w_last_t8)).filter(F.col("rn") == 1)
    .select(*cols(), F.to_date(F.lit("9999-12-31")).alias("event_date"))
)

# --- Step 5: the SCD Type 8 dimension -------------------------------------------
dim_t8 = (
    initial_rows_t8.unionByName(fact_rows_t8).unionByName(current_rows_t8)
    .orderBy(KEY, "event_date")
)

n_states  = states_t8.count()          # what SCD 2 would store: one row per version
n_initial = initial_rows_t8.count()
n_facts   = fact_rows_t8.count()
n_current = current_rows_t8.count()
n_rows_t8 = dim_t8.count()
n_sept    = dim_t8.filter((F.year("event_date") == 2024) & (F.month("event_date") == 9)).count()

print(f"\nSCD 2 row count for the same data:  {n_states} version rows")
print(f"SCD 8 row count:                   {n_rows_t8} rows "
      f"= {n_initial} initial (1900-01-01) + {n_facts} fact-date + {n_current} current (9999-12-31)")
print(f"  -> {n_rows_t8 / n_states:.1f}x more rows than SCD 2 — disk space traded for read speed")
print(f"A September query prunes to:        {n_sept} of {n_rows_t8} rows "
      f"({n_rows_t8 / max(n_sept, 1):.1f}x fewer rows read)")

# the active entity with the most versions shows the richest timeline
T8_SAMPLE_KEY = (
    states_t8.join(deleted_ids_t8, KEY, "left_anti")
    .groupBy(KEY).count().orderBy(F.col("count").desc(), KEY).first()[0]
)
print(f"\n{KEY} = {T8_SAMPLE_KEY} — the full SCD 8 picture: initial row, fact-date rows, current row:")
display(dim_t8.filter(F.col(KEY) == T8_SAMPLE_KEY))

print("\nThe article's typical query: September sales joined on the EQUALITY")
print("sale_date = event_date (no BETWEEN, no valid_from / valid_to):")
sept_facts = (
    sales.filter((F.year("sale_date") == 2024) & (F.month("sale_date") == 9))
    .select(KEY, "sale_date").distinct()
)
display(
    sept_facts.alias("f")
    .join(dim_t8.alias("d"),
          (F.col(f"f.{KEY}") == F.col(f"d.{KEY}")) & (F.col("f.sale_date") == F.col("d.event_date")))
    .select(F.col(f"f.{KEY}"), F.col("f.sale_date"), *[F.col(f"d.{a}") for a in ATTRS])
    .orderBy(F.col(f"f.{KEY}"))
)

# --- Inputs for the MERGE demo (section 8) -------------------------------------
dim_t8.createOrReplaceTempView("dim_t8_v")
active_keys = [r[0] for r in current_rows_t8.select(KEY).orderBy(KEY).collect()]
NEW_SALE_KEYS = active_keys[:2]                       # sold again on a brand-new date
NEW_SALE_DATE = date(2025, 1, 15)

# (a) new fact batch: the 2 entities sold on 2025-01-15 with their current values -> WHEN NOT MATCHED
new_sales_t8 = (
    current_rows_t8.filter(F.col(KEY).isin(NEW_SALE_KEYS))
    .select(*cols(), F.lit(NEW_SALE_DATE).alias("sale_date"))
)

# (b) a data-quality CORRECTION for one entity on an EXISTING fact date (2024-10-01):
#     the source fixed a wrong value that applies to that date only -> WHEN MATCHED update
STRING_ATTRS = [a for a in ATTRS if col_type(a) == "string"]
CORRECTION_DATE = date(2024, 10, 1)
correction_t8 = None
if STRING_ATTRS:
    CORRECTION_ATTR = STRING_ATTRS[-1]
    correction_t8 = (
        fact_rows_t8.filter((F.col("event_date") == F.lit(CORRECTION_DATE)) & ~F.col(KEY).isin(NEW_SALE_KEYS))
        .orderBy(KEY).limit(1)
        .withColumn(CORRECTION_ATTR, F.concat(F.col(CORRECTION_ATTR), F.lit(" (corrected)")))
        .select(*cols(), F.col("event_date").alias("sale_date"))
    )
sales_2025 = new_sales_t8 if correction_t8 is None else new_sales_t8.unionByName(correction_t8)
sales_2025.createOrReplaceTempView("sales_2025_v")

# (c) a LATE source change for the second entity: one attribute changes, effective after
#     its last known version (2024-12-20 for the product sample) -> SCD 1 row + later fact rows
LATE_KEY = NEW_SALE_KEYS[-1]
late_row = current_rows_t8.filter(F.col(KEY) == LATE_KEY).first().asDict()
late_from = states_t8.filter(F.col(KEY) == LATE_KEY).agg(F.max("state_from")).first()[0]
LATE_CHANGE_DATE = max(date(2024, 12, 20), late_from + timedelta(days=1))
LATE_ATTR, LATE_VALUE = SAMPLE.get("late_change") or (TYPE3_ATTR, None)
if LATE_VALUE is None:   # any other value the attribute takes elsewhere in the dimension
    LATE_VALUE = (states_t8.filter(~F.col(LATE_ATTR).eqNullSafe(F.lit(late_row[LATE_ATTR])))
                  .select(LATE_ATTR).orderBy(LATE_ATTR).first()[0])
late_row[LATE_ATTR] = LATE_VALUE
late_change_t8 = (
    spark.createDataFrame([tuple(late_row[c] for c in cols())], BASE_SCHEMA)
    .withColumn("change_date", F.lit(LATE_CHANGE_DATE))
)
late_change_t8.createOrReplaceTempView("late_change_t8_v")

print(f"\nViews ready for section 8: dim_t8_v, sales_2025_v ({sales_2025.count()} rows), late_change_t8_v "
      f"({KEY} = {LATE_KEY}: {LATE_ATTR} -> {LATE_VALUE!r} from {LATE_CHANGE_DATE})")

## 7. Concept of SCD Type 9 — embedded history (`_scd` array)


### Basic characteristics of the Slowly Changing Dimensions type 9 (this notebook's concept):

- Name follows the next available number of the type — and, when rotated, also means "infinity"
- Designed to take advantage of the "Data Lakehouse" architecture
- One row to store the current record and its history
    - Simple navigation from the current context of a given entity
    - "Current" reporting happens without even touching the `_scd` column — column pruning applied in most cases (getting current values)
    - Previous values reached simply via higher-order functions and/or `explode()` to get the history
    - Present row with current values (no flag `is_current`) — historical entries present in ARRAY `_scd`
    - Deletes -> the row is kept for historical reference: the last `_scd` version is closed (`valid_to` < 9999-12-31), and that closed last version — not a flag column — is what marks the entity as deleted
- Uses a `_scd` ARRAY of STRUCT-type elements that contains the history records (no separate history table object in Lakehouse)
    - Each version of a record is an inner ARRAY element holding that version of the record (starting from version 0) — e.g. an element with the attributes, the change version (0-based), and the VALID_FROM / VALID_TO date fields, all without creating separate rows (unlike, say, SCD Type 2)
    - One shared schema (all `_scd` elements share a single STRUCT type: a field added later appears as NULL in older elements, and a version cannot have its own set of fields)
- The schema of the `_scd` field is a free canvas:
    - A theoretical model can define recommended columns
    - "Infinity" is achieved here through the flexibility of this schema — you can add whatever you consider needed
- The remaining columns in the table are the attributes that may change
- You can use higher-order functions to observe a given business phenomenon over time, according to the attribute values current at each moment
- Each change is an append of a record to the `_scd` object plus an in-place update of the attributes directly in the table


### Type 9 — embedded history (single row with `_scd` array)

One row per entity. Current attribute values live in the regular columns (overwritten in place — Type 1 style). The full version history is embedded in an `_scd` `ARRAY<STRUCT>` column — each element holds one version with its `valid_from` / `valid_to` and the attribute values that were current at that time.

* inserts → new row with `_scd` = [version 0]
* updates → current columns overwritten (Type 1) and a new version appended to `_scd` (version 1, 2, …)
* deletes → last version in `_scd` closed (`valid_to` set); row kept for historical reference — no flag column, the closed last version (`valid_to` < 9999-12-31) is what marks the delete

"Current" reporting never touches `_scd` (column pruning). History is reached via higher-order functions and/or explode(). No `is_current` / `is_active` flag — the regular columns always hold the latest values.


In [0]:
# --- Step 0: unify original rows and change events into one stream ---
# Original rows become "initial" events dated 2024-01-01.
all_events = events_with_initial.withColumnRenamed("change_date", "event_date")

# --- Step 1: build per-entity version rows (non-delete events only) ---
# Each version gets a sequential number (0-based) and a valid_from /
# valid_to window derived from the NEXT event for the same entity.
w = Window.partitionBy(KEY).orderBy("event_date")
versions = (
    all_events
    .filter(F.col("change_type") != "delete")
    .withColumn("version", F.row_number().over(w) - 1)
    .withColumn("next_date", F.lead("event_date").over(w))
    .withColumn("valid_from", F.col("event_date"))
    .withColumn(
        "valid_to",
        F.coalesce(F.date_sub(F.col("next_date"), 1), F.to_date(F.lit("9999-12-31"))),
    )
)

# --- Step 2: close the last version for deleted entities ---
# If an entity was deleted, its last version's valid_to is set to
# delete_date - 1 instead of 9999-12-31.
delete_dates = (
    changes
    .filter(F.col("change_type") == "delete")
    .groupBy(KEY).agg(F.max("change_date").alias("delete_date"))
)
versions = (
    versions
    .join(delete_dates, KEY, "left")
    .withColumn(
        "valid_to",
        F.when(
            F.col("delete_date").isNotNull() & F.col("next_date").isNull(),
            F.date_sub(F.col("delete_date"), 1),
        ).otherwise(F.col("valid_to")),
    )
    .drop("next_date", "delete_date")
)

# --- Step 3: collect versions into _scd arrays (sorted by version) ---
SCD_META = ["version", "valid_from", "valid_to"]       # technical fields of every _scd element
scd_entries = versions.select(
    KEY,
    F.struct(F.col("version").cast("long").alias("version"),
             *[F.col(c) for c in SCD_META[1:] + ATTRS]).alias("scd_entry"),
)
scd_arrays = (
    scd_entries
    .groupBy(KEY)
    .agg(F.collect_list("scd_entry").alias("_scd"))
    .withColumn("_scd", F.sort_array("_scd"))          # version is the first struct field
)

# --- Step 4: pick the latest version's values as current columns ---
w_latest = Window.partitionBy(KEY).orderBy(F.col("version").desc())
current = (
    versions
    .withColumn("rn", F.row_number().over(w_latest))
    .filter(F.col("rn") == 1)
    .select(*cols())
)

# --- Step 5: assemble the final Type 9 dimension ---
# No flag column: a deleted entity is recognisable by its last _scd
# version being closed (valid_to < 9999-12-31).
dim_t9_final = current.join(scd_arrays, KEY).orderBy(KEY)

display(dim_t9_final.orderBy(KEY))

# Current snapshot — the regular columns always hold the latest values.
# Without a flag column, deleted entities are recognised via the LAST
# _scd version's valid_to (9999-12-31 = still active) — a higher-order
# function used only in the filter, so the output columns never show _scd.
last_valid_to = F.expr("element_at(_scd, size(_scd)).valid_to")
display(
    dim_t9_final
    .filter(last_valid_to == F.to_date(F.lit("9999-12-31")))
    .select(*cols())
    .orderBy(KEY)
)

# History via explode — every version on its own row
display(
    dim_t9_final
    .select(KEY, F.explode(F.col("_scd")).alias("v"))
    .select(KEY, *[F.col(f"v.{c}") for c in SCD_META],
            *[F.col(f"v.{a}").alias(f"hist_{a}") for a in ATTRS])
    .orderBy(KEY, "version")
)

## 8. Sample with the use of MERGE function in Data Lakehouse

### Setup — MERGE examples (Delta Lake)

`MERGE INTO` works on Delta tables only, so each example below first creates its target table in the schema configured in section 0 (parameters) and then applies the incoming batch with one or more MERGE statements. The DataFrames from the cells above are registered as views so the SQL cells can use them as MERGE sources.


In [0]:
dim_base.createOrReplaceTempView("dim_base_v")
changes.createOrReplaceTempView("changes_v")

K = q(KEY)                  # the business key, quoted for SQL
SHOW_SQL = False            # True: print every generated statement before it runs

def run_sql(statement):
    if SHOW_SQL:
        print(statement)
    return spark.sql(statement)

# the latest event per entity — the source of the Type 1 / 3 / 4 / 5 MERGEs
LATEST_EVENT_SQL = f"""
SELECT * FROM (
  SELECT *, ROW_NUMBER() OVER (PARTITION BY {K} ORDER BY change_date DESC) AS rn
  FROM changes_v
) WHERE rn = 1"""

print("Views ready: dim_base_v, changes_v")

### SCD Type 1 — overwrite via MERGE

- `delete` → `WHEN MATCHED ... THEN DELETE`
- `update` → `WHEN MATCHED ... THEN UPDATE SET` (overwritten in place)
- `insert` → `WHEN NOT MATCHED ... THEN INSERT`


In [0]:
def type1_load(table):
    """Create a Type 1 table from the snapshot and apply the batch (latest event per entity)."""
    run_sql(f"CREATE OR REPLACE TABLE {table} AS SELECT {sql_cols(cols())} FROM dim_base_v")
    run_sql(f"""
MERGE INTO {table} AS tgt
USING ({LATEST_EVENT_SQL}) AS src
  ON {sql_on([KEY])}
WHEN MATCHED AND src.change_type = 'delete' THEN DELETE
WHEN MATCHED AND src.change_type = 'update' THEN UPDATE SET
    {sql_set(ATTRS)}
WHEN NOT MATCHED AND src.change_type != 'delete' THEN
  INSERT ({sql_cols(cols())})
  VALUES ({sql_cols(cols(), 'src.')})""")

type1_load(TBL_T1)
display(spark.table(TBL_T1).orderBy(KEY))

### SCD Type 2 — add new row via MERGE

MERGE can close the old version, but a new version is a brand-new row that does not exist in the target yet — so the classic Delta pattern is: one MERGE to expire old rows + one insert for new rows.


In [0]:
def type2_load(table, current_attrs=()):
    """Create and load a Type 2 table; current_attrs adds Type 1 current_<attr> columns (Type 6).

    The surrogate key is an IDENTITY column: Delta generates a unique key for every version
    row, in this and in every future batch (no hand-made key arithmetic that would restart,
    and collide, with the next incremental load)."""
    cur_cols = [current_col(a) for a in current_attrs]
    cur_ddl  = "".join(f",\n  {q(current_col(a))} {col_type(a)}" for a in current_attrs)
    cur_vals = sql_cols(current_attrs)        # a new version's current_<attr> = its own value
    ins_cols = sql_cols(cols(*SCD_COLS, *cur_cols))

    run_sql(f"""
CREATE OR REPLACE TABLE {table} (
  {sql_ddl(cols())},
  {q(SK_COL)} BIGINT GENERATED ALWAYS AS IDENTITY,
  valid_from DATE,
  valid_to   DATE,
  is_current BOOLEAN{cur_ddl}
)""")

    # initial versions from the snapshot
    run_sql(f"""
INSERT INTO {table} ({ins_cols})
SELECT {sql_list(sql_cols(cols()), "DATE'2024-01-01'", "DATE'9999-12-31'", "TRUE", cur_vals)}
FROM dim_base_v""")

    # MERGE: close the current version of every updated / deleted entity
    run_sql(f"""
MERGE INTO {table} AS tgt
USING (
  SELECT {K}, change_date FROM (
    SELECT {K}, change_date,
           ROW_NUMBER() OVER (PARTITION BY {K} ORDER BY change_date ASC) AS rn
    FROM changes_v
    WHERE change_type IN ('update', 'delete')
  ) WHERE rn = 1
) AS src
  ON {sql_on([KEY])} AND tgt.is_current = true
WHEN MATCHED THEN UPDATE SET
  tgt.is_current = false,
  tgt.valid_to   = DATE_SUB(src.change_date, 1)""")

    # INSERT: new versions for updates, first versions for inserts, with proper windows
    run_sql(f"""
INSERT INTO {table} ({ins_cols})
WITH event_stream AS (
  SELECT {sql_cols(cols())}, change_type, change_date,
         LEAD(change_date) OVER (PARTITION BY {K} ORDER BY change_date) AS next_date
  FROM changes_v
)
SELECT {sql_list(sql_cols(cols()), "change_date",
                 "COALESCE(DATE_SUB(next_date, 1), DATE'9999-12-31')", "next_date IS NULL", cur_vals)}
FROM event_stream
WHERE change_type IN ('insert', 'update')""")

type2_load(TBL_T2)
display(spark.table(TBL_T2).orderBy(KEY, "valid_from"))

### SCD Type 3 — add new column via MERGE

`previous_category` keeps the old value; category takes the new one. `UPDATE SET` can read the target's pre-update value (tgt.category).


In [0]:
A, P = q(TYPE3_ATTR), q(PREV_COL)

run_sql(f"""
CREATE OR REPLACE TABLE {TBL_T3} AS
SELECT {sql_cols(cols())}, CAST(NULL AS {col_type(TYPE3_ATTR)}) AS {P}
FROM dim_base_v""")

# Source: one row per entity = its LATEST event, plus the last value of the tracked
# attribute that DIFFERED from the latest one, computed over the WHOLE event stream
# (initial snapshot + every change in the batch). Collapsing the batch to the latest
# event alone would lose changes that happen inside the batch (e.g. an entity
# inserted and changed again in the same feed).
run_sql(f"""
MERGE INTO {TBL_T3} AS tgt
USING (
  WITH all_events AS (
    SELECT {K}, {A}, DATE'2024-01-01' AS change_date FROM dim_base_v
    UNION ALL
    SELECT {K}, {A}, change_date FROM changes_v WHERE change_type != 'delete'
  ),
  latest AS ({LATEST_EVENT_SQL}),
  prev AS (
    SELECT s.{K}, MAX_BY(s.{A}, s.change_date) AS {P}
    FROM all_events s
    JOIN latest l ON s.{K} = l.{K}
    WHERE NOT (s.{A} <=> l.{A})
    GROUP BY s.{K}
  )
  SELECT l.*, p.{P}
  FROM latest l
  LEFT JOIN prev p ON l.{K} = p.{K}
) AS src
  ON {sql_on([KEY])}
WHEN MATCHED AND src.change_type = 'delete' THEN DELETE
WHEN MATCHED AND src.change_type = 'update' THEN UPDATE SET
    -- the previous value moves only when the TRACKED attribute really changed: an
    -- update of any other attribute must not overwrite the real previous value
    tgt.{P} = COALESCE(src.{P},
                       CASE WHEN NOT (src.{A} <=> tgt.{A}) THEN tgt.{A} ELSE tgt.{P} END),
    {sql_set(ATTRS)}
WHEN NOT MATCHED AND src.change_type != 'delete' THEN
  INSERT ({sql_cols(cols(PREV_COL))})
  VALUES ({sql_cols(cols(PREV_COL), 'src.')})""")

display(spark.table(TBL_T3).orderBy(KEY))

### SCD Type 4 — mini-dimension via MERGE

Two tables: a mini-dimension (one row per attribute combination) and a base dimension holding stable attributes + the current `mini_sk` reference. A single MERGE handles all three change kinds on the base table: re-point the reference, delete the base row, insert new rows.


### Mini-dimension

The mini-dimension holds every combination of `category`, `brand` and `color` that exists now or arrives in the batch. Surrogate keys are generated with `DENSE_RANK`, because `MERGE` cannot generate new keys for the rows it inserts.


In [0]:
def mini_dim_load(mini_table):
    """Every combination of the volatile attributes that exists now or arrives in the batch.
    Keys come from DENSE_RANK, because MERGE cannot generate new keys for the rows it inserts."""
    VA = sql_cols(VOLATILE_ATTRS)
    run_sql(f"""
CREATE OR REPLACE TABLE {mini_table} AS
SELECT {VA}, CAST(1000 + DENSE_RANK() OVER (ORDER BY {VA}) AS BIGINT) AS mini_sk
FROM (
  SELECT {VA} FROM dim_base_v
  UNION
  SELECT {VA} FROM changes_v WHERE change_type IN ('insert', 'update')
) AS combos""")

def mini_dim_base_load(base_table, mini_table, outrigger_attrs=()):
    """Base dimension (stable attributes + mini_sk [+ Type 1 outrigger copies]) and its MERGE."""
    out_cols = [current_col(a) for a in outrigger_attrs]
    out_sel  = ", ".join(f"m.{q(a)} AS {q(current_col(a))}" for a in outrigger_attrs)
    run_sql(f"""
CREATE OR REPLACE TABLE {base_table} AS
SELECT {sql_list(f'd.{K}', sql_cols(STABLE_ATTRS, 'd.'), 'm.mini_sk', out_sel)}
FROM dim_base_v AS d
JOIN {mini_table} AS m ON {sql_on(VOLATILE_ATTRS, 'd', 'm', null_safe=True)}""")

    run_sql(f"""
MERGE INTO {base_table} AS tgt
USING (
  SELECT {sql_list('s.*', 'm.mini_sk', out_sel)}
  FROM ({LATEST_EVENT_SQL}) AS s
  LEFT JOIN {mini_table} AS m            -- LEFT: delete rows carry no attributes
    ON {sql_on(VOLATILE_ATTRS, 's', 'm', null_safe=True)}
) AS src
  ON {sql_on([KEY])}
WHEN MATCHED AND src.change_type = 'delete' THEN DELETE
WHEN MATCHED AND src.change_type = 'update' THEN UPDATE SET
    {sql_set(['mini_sk', *out_cols])}
WHEN NOT MATCHED AND src.change_type != 'delete' THEN
  INSERT ({sql_cols([KEY, *STABLE_ATTRS, 'mini_sk', *out_cols])})
  VALUES ({sql_cols([KEY, *STABLE_ATTRS, 'mini_sk', *out_cols], 'src.')})""")

mini_dim_load(TBL_T4_MINI)
mini_dim_base_load(TBL_T4_BASE, TBL_T4_MINI)

display(spark.table(TBL_T4_MINI).orderBy("mini_sk"))
display(spark.table(TBL_T4_BASE).orderBy(KEY))

### SCD Type 5 — mini-dimension + Type 1 outrigger via MERGE

Same two tables as Type 4, but the base dimension also carries a Type 1 copy (outrigger) of the mini-dimension attributes — updated in the same MERGE that re-points the `mini_sk` reference.


In [0]:
# Same two tables as Type 4, but the base dimension also carries a Type 1 copy (outrigger)
# of the mini-dimension attributes — updated in the same MERGE that re-points mini_sk.
mini_dim_load(TBL_T5_MINI)
mini_dim_base_load(TBL_T5_BASE, TBL_T5_MINI, outrigger_attrs=OUTRIGGER_ATTRS)

display(spark.table(TBL_T5_BASE).orderBy(KEY))

### SCD Type 6 — hybrid (Type 2 + Type 1 current-value column) via MERGE

Three steps: close old versions (MERGE), add new versions (insert), then stamp the latest category onto every historical row (second MERGE). A MERGE source cannot be the target table itself, so the current values are materialized into a small helper table first.


In [0]:
# Type 2 load with the extra current_<attr> column(s) — a new version's current value is its own value
type2_load(TBL_T6, current_attrs=TYPE6_ATTRS)

# Helper table: latest value(s) per entity (a MERGE source cannot be the target itself)
run_sql(f"""
CREATE OR REPLACE TABLE {TBL_T6_CURRENT} AS
SELECT {sql_cols([KEY, *TYPE6_ATTRS])} FROM {TBL_T6} WHERE is_current = true""")

# MERGE 2: stamp the latest value(s) onto EVERY historical row
run_sql(f"""
MERGE INTO {TBL_T6} AS tgt
USING {TBL_T6_CURRENT} AS src
  ON {sql_on([KEY])}
WHEN MATCHED THEN UPDATE SET
    {', '.join(f'tgt.{q(current_col(a))} = src.{q(a)}' for a in TYPE6_ATTRS)}
WHEN NOT MATCHED BY SOURCE THEN UPDATE SET
    {', '.join(f'tgt.{q(current_col(a))} = NULL' for a in TYPE6_ATTRS)}""")

display(spark.table(TBL_T6).orderBy(KEY, "valid_from"))

### SCD Type 7 — dual Type 1 + Type 2 via MERGE

Two physical tables, each maintained with its own MERGE from the same incoming batch: a current snapshot table (Type 1) and a full history table (Type 2).


In [0]:
# Two physical tables, each maintained from the same incoming batch by the loaders above
type1_load(TBL_T7_CURRENT)      # Type 1 side: overwrite in place / delete / insert
type2_load(TBL_T7_HISTORY)      # Type 2 side: close old versions + add the new ones

display(spark.table(TBL_T7_CURRENT).orderBy(KEY))
display(spark.table(TBL_T7_HISTORY).orderBy(KEY, "valid_from"))

### SCD Type 8 — fact-date rows via MERGE (Oleg Ivanov, 2017)

The dimension is seeded from the simulated sales fact: every `(product_id, sale_date)` combination becomes one row with the attributes as of that date, plus the special-date rows — `1900-01-01` (initial state, SCD 0) and `9999-12-31` (current state, SCD 1). The table is **partitioned by `event_date`**, the way the article prescribes, so a month-bounded query reads one partition (the next cell shows the partitions).

* new sale dates → `WHEN NOT MATCHED ... THEN INSERT` (a new fact-date row)
* a data-quality correction on an existing fact date (it applies to that date only) → `WHEN MATCHED ... THEN UPDATE`
* a late source change (effective 2024-12-20) → a second MERGE overwrites the `9999-12-31` row in place (the SCD 1 bonus) **and** every fact-date row dated on or after the change date; fact-date rows before it keep what was true at their date

No `valid_from` / `valid_to`, no `is_current`, no surrogate key: the match key is the pair `(product_id, event_date)` itself.

In [0]:
# Seed: initial rows (1900-01-01) + one row per (entity, sale date) +
# current rows (9999-12-31) — partitioned by event_date, like the fact table
run_sql(f"""
CREATE OR REPLACE TABLE {TBL_T8}
PARTITIONED BY (event_date)
AS SELECT * FROM dim_t8_v""")

# MERGE 1: new sales arrive (2025-01-15) + one CORRECTED row:
# the correction is a data-quality fix that applies to its own date only
run_sql(f"""
MERGE INTO {TBL_T8} AS tgt
USING sales_2025_v AS src
  ON {sql_on([KEY])} AND tgt.event_date = src.sale_date
WHEN MATCHED THEN UPDATE SET
    {sql_set(ATTRS)}
WHEN NOT MATCHED THEN
  INSERT ({sql_cols(cols('event_date'))})
  VALUES ({sql_cols(cols(), 'src.')}, src.sale_date)""")

# MERGE 2: a late source change. The 9999-12-31 row is overwritten in place (the SCD 1
# bonus), and so is every fact-date row dated ON OR AFTER the change date (the new value
# was already true then). Fact-date rows BEFORE it keep the value true at their date.
run_sql(f"""
MERGE INTO {TBL_T8} AS tgt
USING late_change_t8_v AS src
  ON {sql_on([KEY])} AND tgt.event_date >= src.change_date
WHEN MATCHED THEN UPDATE SET
    {sql_set(ATTRS)}""")

# The late-changed entity: initial row, fact-date rows (2025-01-15 restated), overwritten 9999-12-31 row
display(spark.table(TBL_T8).filter(F.col(KEY) == LATE_KEY).orderBy("event_date"))

In [0]:
# One partition per event_date — including the two special dates.
# This is what makes the article's partition pruning work: a query bounded
# to one month of facts reads exactly one of these partitions.
display(spark.sql(f"SHOW PARTITIONS {TBL_T8}"))

### SCD Type 9 — embedded history via MERGE

One row per entity. Current values in regular columns (Type 1), full version history embedded in an `_scd` `ARRAY<STRUCT>` column.

With multiple change events per product a single MERGE cannot append multiple versions in one pass (each target row matches at most one source row). Instead we pre-compute the complete `_scd` array per product, load it via `CREATE OR REPLACE TABLE`, then demonstrate the incremental MERGE pattern for a single new change.


In [0]:
# --- Build the complete final state from the source data ---
run_sql(f"""
CREATE OR REPLACE TABLE {TBL_T9} AS
WITH all_events AS (
  SELECT {sql_cols(cols())}, 'initial' AS change_type, DATE'2024-01-01' AS event_date
  FROM dim_base_v
  UNION ALL
  SELECT {sql_cols(cols())}, change_type, change_date AS event_date
  FROM changes_v
),
versions AS (
  SELECT {sql_cols(cols())},
         ROW_NUMBER() OVER (PARTITION BY {K} ORDER BY event_date) - 1 AS version,
         event_date AS valid_from,
         LEAD(event_date) OVER (PARTITION BY {K} ORDER BY event_date) AS next_date
  FROM all_events
  WHERE change_type != 'delete'
),
closed AS (
  SELECT v.*,
         CASE WHEN d.delete_date IS NOT NULL AND v.next_date IS NULL THEN DATE_SUB(d.delete_date, 1)
              WHEN v.next_date IS NOT NULL THEN DATE_SUB(v.next_date, 1)
              ELSE DATE'9999-12-31'
         END AS valid_to
  FROM versions v
  LEFT JOIN (
    SELECT {K}, MAX(change_date) AS delete_date
    FROM changes_v WHERE change_type = 'delete' GROUP BY {K}
  ) d ON v.{K} = d.{K}
),
scd_arrays AS (
  SELECT {K},
         ARRAY_SORT(COLLECT_LIST(named_struct(
           'version', CAST(version AS BIGINT), 'valid_from', valid_from, 'valid_to', valid_to,
           {sql_struct(ATTRS)}))) AS _scd
  FROM closed
  GROUP BY {K}
),
current_vals AS (
  SELECT {sql_cols(cols())} FROM (
    SELECT *, ROW_NUMBER() OVER (PARTITION BY {K} ORDER BY version DESC) AS rn FROM closed
  ) WHERE rn = 1
)
SELECT {sql_cols(cols(), 'c.')}, s._scd
FROM current_vals c
JOIN scd_arrays s ON c.{K} = s.{K}""")

# --- Incremental MERGE: add a single brand-new entity (demonstration) ---
# For real incremental loads, process one change batch at a time
# (at most one event per entity per batch).
(dim_t1.orderBy(KEY).limit(1)                       # attributes borrowed from an existing entity
 .withColumn(KEY, F.lit(new_key(0)).cast(KEY_TYPE))
 .withColumn("change_date", F.lit(date(2025, 1, 15)))
 .createOrReplaceTempView("t9_new_entity_v"))

run_sql(f"""
MERGE INTO {TBL_T9} AS tgt
USING t9_new_entity_v AS src
  ON {sql_on([KEY])}
WHEN NOT MATCHED THEN
  INSERT ({sql_cols(cols('_scd'))})
  VALUES ({sql_cols(cols(), 'src.')},
          ARRAY(named_struct('version', CAST(0 AS BIGINT), 'valid_from', src.change_date,
                             'valid_to', DATE'9999-12-31', {sql_struct(ATTRS, 'src.')})))""")

display(spark.table(TBL_T9).orderBy(KEY))

# History exploded — every version on its own row
display(
    spark.table(TBL_T9)
    .select(KEY, F.explode("_scd").alias("hist"))
    .select(KEY, "hist.*")
    .orderBy(KEY, "version")
)

## 9. Point-in-time query — SCD Type 9

Explode the `_scd` array and filter on `valid_from` / `valid_to` to reconstruct the dimension table as it looked at any chosen date. No separate history table, no complex join — just column access into the embedded history.

### Point-in-time query — "what did the dimension look like on date X?"

SCD Type 9's `_scd` array makes point-in-time queries trivial: explode the `_scd` array, filter to the version whose `valid_from` / `valid_to` window contains the target date, and you get the exact attribute values every product had at that moment.

We pick 2024-09-15 — a date that falls between several change waves, so products are at different versions (some at v0, some at v1, v2, …).


In [0]:
target_date = "2024-09-15"

point_in_time = (
    dim_t9_final
    .select(KEY, F.explode(F.col("_scd")).alias("v"))
    .filter(
        (F.col("v.valid_from") <= F.lit(target_date).cast("date"))
        & (F.col("v.valid_to") >= F.lit(target_date).cast("date"))
    )
    .select(KEY, "v.version", *[F.col(f"v.{a}") for a in ATTRS], "v.valid_from", "v.valid_to")
    .orderBy(KEY)
)

print(f"Dimension state on {target_date}:")
display(point_in_time)

# 10. Slowly Changing Dimensions (SCD) - comparison of announced Types with the concept one

In dimensional modeling, a **slowly changing dimension** is a dimension whose attribute values change over time — slowly, not every transaction. Ralph Kimball defined six classic types (0–6); a seventh is commonly added in practice; an eighth (fact-date rows) was introduced by Oleg Ivanov in 2017; and this notebook adds its own concept — embedded history — as a ninth type. Each type answers the question: *when an attribute value changes, what do we do with the old value?*

---

| Type | Author | Name | Strategy | Year announced | Old value preserved? | Full history? | New objects created |
|------|--------|------|---------|---------------|---------------------|---------------|--------------------|
| 0 | Ralph Kimball / Kimball Group | Retain / Original | Never change the value | 1996 (2008*) | ✅ (never overwritten) | ❌ | None — no schema or table changes |
| 1 | Ralph Kimball | Overwrite | Replace old with new | 1996 | ❌ | ❌ | None — same table, same columns, in-place updates |
| 2 | Ralph Kimball | Add New Row | Insert a new row, close out the old | 1996 | ✅ | ✅ | **New columns** added to the existing dimension: `valid_from`, `valid_to`, `is_current`, plus a **surrogate key** column (e.g. `product_sk`) |
| 3 | Ralph Kimball | Add New Column | Keep old + new in separate columns | 1996 | ✅ (one prior version) | Partial (current + previous) | **New columns** added to the existing dimension (e.g. `previous_category`) — one per tracked attribute |
| 4 | Ralph Kimball | Mini-Dimension | Rapidly changing attrs split into a separate dimension | 2002 | N/A (separate table) | ✅ (via mini-dim grouping) | **New table** (mini-dimension) for the rapidly-changing attributes, keyed by a surrogate key; base dimension gains a **new FK column** (e.g. `mini_sk`) pointing to it |
| 5 | Ralph Kimball | Mini-Dimension + Type 1 Outrigger | Type 4 + Type 1 ref in base dim | 2002 | Partial | ✅ (via mini-dim) | Everything from Type 4 **plus new outrigger columns** copied into the base dimension (e.g. `current_category`, `current_brand`) carrying the latest mini-dim values |
| 6 | Ralph Kimball | Hybrid (1 + 2) | Type 2 history + Type 1 overwrite for selected attrs | 2002 | Partial | ✅ | All Type 2 columns **plus a new current-value column** (e.g. `current_category`) stamped across every history row |
| 7 | Kimball Group | Dual Type (1 + 2) | Maintain both a Type 1 table and a Type 2 table | 2013 | Partial | ✅ | **Second physical table** — a Type 1 snapshot table alongside the Type 2 history table (two tables maintained in parallel) |
| 8 | Oleg Ivanov | Fact-Date Rows | One row per entity × fact date; single `event_date` replaces `DATE_FROM`/`DATE_TO`; partitioned by `event_date` | 2017 | ✅ (value at each fact date) | ✅ (aligned to the facts) | No schema change — the *rows* multiply with the facts; special dates `1900-01-01` (initial) / `9999-12-31` (current) combine SCD 0 + SCD 1 in the same table |
| 9 | Bartosz Malec | Embedded History (∞) — concept | One row per entity; full history in `ARRAY<STRUCT>` `_scd` column | 2026 (this notebook) | ✅ | ✅ | **New column** `_scd` (`ARRAY<STRUCT>`) added to the existing dimension — no new tables; each array element holds one version with attributes, version number, and `valid_from`/`valid_to` |

---

*announced in 1996 with other concepts, but described in details in 2008.

## Detailed explanations

### Type 0 — Retain (Original Value)
Never modify the attribute — the value set at insert time is permanent, regardless of later changes in the source. Rarely used, but useful for immutable facts like a customer's original registration date or date of birth.

> **New objects:** None. The dimension table and its schema stay exactly as-is; no columns, tables, or lookup structures are added.
>
> **Announced:** Formally described by the Kimball Group in the third edition of *The Data Warehouse Toolkit* (2008), complementing the original three types.
> **Data Warehouse use case:** Preserving immutable attributes such as a customer's original registration date or date of birth that must never be modified.
> **Data Lakehouse use case:** Retaining original source-system landing timestamps in the Bronze layer for full audit traceability across the medallion architecture.

### Type 1 — Overwrite
Simply update the row in place. No history is kept — the old value is lost. This is the simplest and most common approach when only the current value matters (e.g., correcting a misspelled name).

> **New objects:** None. Same table, same columns — rows are simply overwritten in place.
>
> **Announced:** Introduced by Ralph Kimball in the first edition of *The Data Warehouse Toolkit* (1996) — one of the three original SCD types.
> **Data Warehouse use case:** Correcting a misspelled customer name or keeping the latest shipping address when only the current value matters.
> **Data Lakehouse use case:** Overwriting the latest record state in the Silver/Gold Delta tables via `MERGE` for downstream consumers that need only current values.

### Type 2 — Add New Row (Historical Tracking)
Insert a **new row** for the changed record and mark the previous row as expired. This preserves the full history of changes. Each row gets:
* **effective dates** (or timestamps): `valid_from` / `valid_to`
* a **current flag** (`is_current` = true/false)
* a **surrogate key** so every version has a unique primary key

This is the most widely used SCD type in data warehousing because it lets fact rows join to the dimension version that was active at the time of the fact.

> **New objects:** The **existing dimension table gains new columns** — at minimum a surrogate key (e.g. `product_sk`), `valid_from`, `valid_to`, and `is_current`. No additional tables are created.
>
> **Announced:** Introduced by Ralph Kimball in the first edition of *The Data Warehouse Toolkit* (1996) — one of the three original SCD types.
> **Data Warehouse use case:** Tracking full customer address history so that fact rows join to the dimension version that was active at the time of each transaction.
> **Data Lakehouse use case:** Maintaining full version history in Delta tables using Change Data Feed (CDF) with `MERGE` SCD Type 2 logic for point-in-time analytics and incremental CDC pipelines.

### Type 3 — Add New Column
Add a second column to hold the prior value (e.g., `previous_category` alongside `category`). Only one level of history is kept, so this is suitable when you only need to compare the current value to the immediately preceding one.

> **New objects:** **New columns** added to the existing dimension table — one `previous_*` column per tracked attribute (e.g. `previous_category`). No additional tables are created.
>
> **Announced:** Introduced by Ralph Kimball in the first edition of *The Data Warehouse Toolkit* (1996) — one of the three original SCD types.
> **Data Warehouse use case:** Comparing current vs. previous sales-territory assignment for an employee without the overhead of full history rows.
> **Data Lakehouse use case:** Lightweight one-level history tracking in a Silver Delta table when full SCD Type 2 is unnecessary but a before/after comparison is still required.

### Type 4 — Mini-Dimension
When certain attributes change **rapidly** (e.g., number of dependents, income band, credit score), they are split out of the main dimension into a separate *mini-dimension*. Each unique combination of those attributes becomes a row in the mini-dimension, keyed by a surrogate key. The base dimension references the current mini-dimension key (a Type 1 overwrite on that key). Fact rows then reference the mini-dimension surrogate key, capturing history without exploding the base dimension.

> **New objects:** A **new lookup table** (the mini-dimension) holding one row per unique attribute combination, keyed by its own surrogate key. The base dimension gains a **new foreign-key column** (e.g. `mini_sk`) referencing the mini-dimension.
>
> **Announced:** Introduced in the second edition of *The Data Warehouse Toolkit* (2002) to handle rapidly changing attributes.
> **Data Warehouse use case:** Managing fast-changing customer demographics (income band, credit score, dependents) without exploding the base customer dimension.
> **Data Lakehouse use case:** Splitting volatile attributes into a separate Delta table keyed by surrogate key, referenced from the main dimension to keep the base table compact and queryable.

### Type 5 — Mini-Dimension + Type 1 Outrigger
A Type 4 mini-dimension, plus the **base dimension also carries a Type 1 copy** of the key mini-dimension attributes as an outrigger (overwritten on change). This lets you group or filter by the *current* value of a rapidly changing attribute directly from the base dimension, while the mini-dimension preserves the full history.

> **New objects:** Everything from Type 4 (new mini-dimension table + `mini_sk` FK on the base) **plus new outrigger columns** on the base dimension (e.g. `current_category`, `current_brand`) that carry a Type 1 copy of the latest mini-dimension attribute values.
>
> **Announced:** Introduced in the second edition of *The Data Warehouse Toolkit* (2002) as an extension of the Type 4 mini-dimension pattern.
> **Data Warehouse use case:** Allowing BI dashboards to filter by the current income band directly from the base dimension while the mini-dimension preserves full historical accuracy.
> **Data Lakehouse use case:** Adding current-value outrigger columns in the Gold layer so BI tools can query volatile attributes without joining to the mini-dimension Delta table.

### Type 6 — Hybrid (Type 1 + Type 2)
Combines Type 2 (full history rows with effective dates) **and** Type 1 (overwrite) for a subset of columns. Typical implementation: the dimension has history rows (Type 2), but also maintains a **current-value column** that is always overwritten to the latest value (Type 1) across *all* rows. This lets you group all historical facts by the current value of an attribute, while still preserving the value as it was at each point in time.

> **New objects:** All Type 2 columns (`product_sk`, `valid_from`, `valid_to`, `is_current`) **plus a new current-value column** (e.g. `current_category`) added to the existing dimension table. No additional tables are created.
>
> **Announced:** Introduced in the second edition of *The Data Warehouse Toolkit* (2002) as a hybrid combining Types 1 and 2.
> **Data Warehouse use case:** Grouping all historical sales by the *current* product category while still preserving the category value as it was at each point in time.
> **Data Lakehouse use case:** Stamping a `current_*` column across every history row in a Delta table so a single Gold-layer query supports both current and point-in-time analytics.

### Type 7 — Dual Type 1 + Type 2
Maintain **two physical tables**: one Type 1 table (current state only, overwritten) and one Type 2 table (full history). Fact tables join to the Type 2 table for point-in-time accuracy, while the Type 1 table serves dashboards and queries that only need the latest snapshot. This trades extra storage for query simplicity.

> **New objects:** A **second physical table** — a Type 1 snapshot table alongside the Type 2 history table. The Type 2 table carries the SCD 2 columns (`product_sk`, `valid_from`, `valid_to`, `is_current`); the Type 1 table mirrors the original dimension schema. Two tables must be maintained and kept in sync.
>
> **Announced:** Type 7 was formally described in 2013 by the Kimball Group. It was included in the 3rd edition of The Data Warehouse Toolkit the same year.
> **Data Warehouse use case:** Serving a Type 1 current-state table for fast dashboards and a Type 2 history table for accurate point-in-time fact joins from the same source feed.
> **Data Lakehouse use case:** Maintaining a current-state Delta table (feeding BI/SQL warehouses) alongside a full-history Delta table (feeding incremental CDC and audit pipelines), both refreshed from the same batch or streaming source.

### Type 8 — Fact-Date Rows (Oleg Ivanov, 2017)
A row is added to the dimension for each combination of entity key and fact date. The `DATE_FROM`/`DATE_TO` construction of Type 2 is replaced with a single `EVENT_DATE`, and the table is partitioned by it — in the same manner as the fact table — so queries bounded by the fact date prune to exactly the needed partitions. The trade is disk space for performance: the article's example dimension grows from 21M (SCD 2) to 120M rows (SCD 8), while a typical one-month query reads 21× fewer rows. Best suited for high-cardinality, often-changed dimensions where a long history is stored but a small window is analyzed at a time and facts are sparse per key.

> **New objects:** None in the schema — the change is in the *rows*: one per entity × fact date, plus "special" dates (`1900-01-01` for initial rows — SCD 0, `9999-12-31` for the last version — SCD 1), so several SCD types coexist in one table.
>
> **Announced:** Proposed by Oleg Ivanov in *"Introducing Slowly Changing Dimension Type 8"* (LinkedIn, 3 April 2017); the author notes the name and number are not yet established.
> **Data Warehouse use case:** Large client/product dimensions that change often and are joined to facts analyzed in short windows (e.g. monthly), where partition pruning beats full SCD 2 scans.
> **Data Lakehouse use case:** A Delta table partitioned (or liquid-clustered) by `event_date`, aligned to the fact table's date; equality joins on `sale_date = event_date` enable partition pruning and precise cardinality estimates; Delta's `MERGE` applies new fact dates and restatements.

### Type 9 — Embedded History (∞) — notebook concept
One row per entity stores **both current and historical data**. Current attribute values live in scalar columns (no `is_current` flag); the full version history is embedded in an `_scd` `ARRAY<STRUCT>` column where each element holds one version with its attributes, version number, and `valid_from` / `valid_to` timestamps. The name "9" also represents infinity (∞) when rotated, reflecting the unlimited schema flexibility of the embedded history field. Designed for the **Data Lakehouse** architecture: column pruning serves current-value queries without touching `_scd`, while higher-order functions and `explode()` access historical versions for point-in-time analysis. Each change appends a record to the `_scd` array and updates the scalar columns in place — no new rows, no separate history table.

> **New objects:** A **new column** `_scd` (`ARRAY<STRUCT>`) added to the existing dimension table. No additional tables are created — the full history lives inside the single row per entity.
>
> **Announced:** Bartosz Malec — this notebook's own concept (2026); the number follows the next available SCD type number after Type 8 and, when rotated, represents infinity (∞).
> **Data Warehouse use case:** Not traditionally applicable — this type was designed specifically for Lakehouse-native patterns.
> **Data Lakehouse use case:** Maintaining one row per entity in a Delta table with embedded history, enabling both current-value queries (via column pruning) and point-in-time analytics (via array functions) without a separate history table or multi-row joins.

# 11. SCD Type 9 vs. Data Vault 2.0 — audit history of records

Both SCD Type 9 and Data Vault 2.0 answer the same core question — *"how do we keep the full history of what an attribute looked like, when it changed, and without ever losing the old value?
"* — but they operate at very different levels of the architecture.

### Data Vault 2.0 — where the audit history lives

Data Vault 2.0 splits a model into **Hubs** (business keys), **Links** (relationships), and **Satellites** (all descriptive attributes + the audit trail). The audit history is carried by 
Satellites:

- **`load_date` / `load_end_date`** — each Satellite row is a "version" of the record, opened and closed exactly like SCD Type 2's `valid_from` / `valid_to`, one row per change event.
- **`hash_diff`** — a hash of the attribute payload that is recomputed on each load; a change is detected by comparing hashes, not column-by-column (mirrors how Delta's `MERGE` can match on 
a hash to decide whether anything changed).
- **`record_source`** — provenance of every version (which system, which feed) — something the `_scd` array does *not* carry by default.
- **Point-in-Time (PIT) tables** and **current-state views** over the Raw Vault reconstruct "what the entity looked like on date X" or "the latest values" — the same jobs done by SCD Type 9's `explode()` on `_scd` and plain column reads respectively.
- The **Business Vault** layer materializes the dimensional/SCD-style outputs (including SCD Type 2 tables) on top of the raw audit history.

### Side-by-side comparison

| Aspect | SCD Type 9 (embedded history) | Data Vault 2.0 (Satellite-based) |
|---|---|---|
| Audit granularity | One row per entity; every change is an element appended to the `ARRAY<STRUCT>` `_scd` column | One row per change event per Satellite; history grows as separate rows (hub–link–satellite structure) |
| Change detection | Per-attribute or hash comparison inside `MERGE` logic | `hash_diff` comparison — built into the standard load pattern |
| Point-in-time query | `explode(_scd)` + filter `valid_from`/`valid_to` in a single table scan of one row per entity | Query across Hub → Satellite joins (or PIT table) — more joins, but the model is pre-joined once in the PIT |
| Current values | Plain scalar columns — column pruning, no history touched; fastest possible read | Current-state views over Satellites (or PIT) — extra view/join hop, but still materialized views |
| Provenance / audit metadata | Schema of `_scd` is a free canvas — version number, `valid_from`/`valid_to` included, but `record_source` and load timestamps are not standardized; you add them if needed | Audit is standardized by design: `load_date`, `hash_diff`, `record_source` on every Satellite row |
| Schema evolution | Elastic at the array level — all elements share ONE struct type, so adding a field widens every element (older versions read it as NULL); a single version cannot have its own set of fields | Satellites evolve by adding columns; old rows keep `NULL` for new attributes |
| Model scope | A single dimension table — a tactical pattern for one table | An entire enterprise modeling methodology — normalized raw layer + agile delivery of marts |
| Storage / update cost | Each change = rewrite of the `_scd` array + in-place attribute update in one row | Each change = simple append of one Satellite row (insert-only Raw Vault loads, parallelizable) |
| Deletes | Must be modeled explicitly (closure of the last `_scd` version's `valid_to`), as done in this notebook | Not modeled in Raw Vault — deletes arrive as new Satellite rows or are resolved in the Business Vault |
| Rebuilding dimensional marts | Direct — the dimension *is* the history | Business Vault / `MERGE` jobs generate SCD 1/2/6/9-style marts from the Vault on demand |

### Takeaways

- **Complementary, not competing.** Data Vault 2.0 is a full modeling methodology whose Satellites are effectively a *de-normalized-at-the-record* audit history; SCD Type 9 is a compact, 
Lakehouse-native way to embed the *resulting* history into one row of the consuming dimension.
- In a Lakehouse, a natural pattern is: **Raw Vault Satellites as the system-of-record audit layer**, then materialize SCD Type 9 dimensions (`_scd` arrays) into the Gold layer from that 
history — you get standardized provenance and audit metadata *and* fast current-value / point-in-time access in the dimensional layer.
- Where SCD Type 9 wins: simplicity (one table, no joins), column pruning for current-value queries, one shared schema that evolves in place, no per-version schemas, and trivial point-in-time via `explode()`.
- Where Data Vault 2.0 wins: standardized audit metadata (`record_source`, `load_date`, `hash_diff`), insert-only parallel loads, scaling to enterprise-wide models with many sources, and 
on-demand generation of *any* SCD type (including Type 9) as a Business Vault deliverable.

# 12. Case: schema evolution — new fields added to the dimension

Schemas are never frozen: one day the source system starts sending a **`supplier`**
attribute, later a **`warranty_months`** one — fields that simply did not exist when
the dimension was first built. This section simulates a **second feed** whose
events introduce new attributes mid-stream:

1. **2025-01-10** — the field `supplier` appears in the source for the first time (updates + a brand-new insert that already carries it)
2. **2025-02-15 / 2025-02-20** — a second new field, `warranty_months`, appears

For **SCD Type 9** the case is executed in full: the feed is applied to the
`dim_t9_final` table from section 7 and the result shows how the `_scd`
column **behaves and evolves** — the `ARRAY<STRUCT>` schema is widened in place:

* every **existing** version element keeps "the fields that were actual at its
  time" — the new fields stay `NULL` for versions created before the field existed
  (schema **elasticity**),
* every **new** version element carries the fields that exist when it is created,
* the regular (current) columns of the table gain the new attributes too —
  schema evolution of the table itself, with no backfill of history and no second
  physical table.

For **Types 0–8 only a summary** is given (next cell): there the new fields force a
flat, table-wide schema change, because all versions share one column set.

### How Types 0–8 handle this schema evolution (summary; Type 9 is executed in full below)

All classic types store versions in **one flat schema** — rows or columns of a
single table — so a new field is always a table-wide change; old versions keep
`NULL` and cannot express "this attribute did not exist back then".

| Type | Author | How the new fields (`supplier`, `warranty_months`) would be handled |
|------|--------|---------------------------------------------------------------------|
| **0 — Retain** | Ralph Kimball / Kimball Group | `ALTER TABLE … ADD COLUMN` once. Existing rows keep `NULL` forever (Type 0 never rewrites anything); only records inserted after the change can carry a value. |
| **1 — Overwrite** | Ralph Kimball | `ALTER TABLE … ADD COLUMN`; existing rows are `NULL` until the next update/insert overwrites them, or a one-off backfill `UPDATE` fills them. A current-state table — history was never kept anyway. |
| **2 — Add new row** | Ralph Kimball | Add the columns to the versioned table. Old version rows keep `NULL` — which here is actually correct: the field did not exist then. New version rows carry values. The load MERGE runs with schema evolution (`spark.databricks.delta.schema.autoMerge.enabled = true`) or an explicit `ALTER TABLE` first. |
| **3 — Add new column** | Ralph Kimball | The new attribute becomes just another current column (`supplier`); if the *previous* value must also be kept, a `previous_supplier` column is added alongside — one more column per tracked attribute, one level of history only. |
| **4 — Mini-dimension** | Ralph Kimball | First decide *where the field belongs*: rapidly changing → the mini-dimension gains the attribute (every combination is re-keyed, `mini_sk` values change); stable → the base dimension gains a plain column. |
| **5 — Mini-dim + Type 1 outrigger** | Ralph Kimball | The Type 4 decision **plus**, for mini-dimension fields, a new outrigger column in the base dimension (e.g. `current_supplier`) carrying the latest mini-dim value. |
| **6 — Hybrid (1 + 2)** | Ralph Kimball | Type 2 handling for history rows (old versions `NULL`, new versions valued) **plus** the new current-value column (`current_supplier`) stamped across *every* history row on the next load. |
| **7 — Dual (1 + 2)** | Kimball Group | Both physical tables must be altered and backfilled **in sync** — double schema change, double backfill, risk of drift between the snapshot and history tables. |
| **8 — Fact-date rows (Ivanov 2017)** | Oleg Ivanov | `ALTER TABLE … ADD COLUMN` once. Only the `9999-12-31` row and future fact-date rows carry the new field; historical fact-date rows keep `NULL` — which is correct: the field did not exist at that date. Partitioning by `event_date` is unaffected. |
| **9 — Embedded history (concept)** | Bartosz Malec | The case executed in full below: the `_scd` struct widens in place — every version keeps the fields that were actual at its time (elasticity), no backfill, no second table. |

**The contrast with Type 9:** in Types 0–8 the schema is a property of the
*table* — one shape for all versions, so a new field is a table-wide `ALTER`.
In Type 9 the schema is a property of the *version*: each `_scd` element keeps
the fields that existed when that version was created, so history is never
rewritten when a new field arrives.

### Schema evolution feed — new fields appear mid-stream

A second change feed arrives. Unlike the 2024 feed, these events carry new attributes that did not exist when the dimension was built:

* 2025-01-10 → "supplier" appears in the source for the first time
* 2025-02-15+ → "`warranty_months`" appears for the first time

Events created before a field existed simply do not carry it (NULL) — exactly how a real source behaves when new columns are switched on. No delete events here — the focus is purely on schema growth.


In [0]:
# A second change feed whose events carry NEW attributes that did not exist when the
# dimension was built. Events created before a field existed do not carry it (NULL).
SAMPLE_EVOLUTION_FEEDS = {
    "product": {
        "fields": [("supplier", "string"), ("warranty_months", "bigint")],
        "events": [   # (key, <attributes>, <new fields>, change_type, change_date)
            # --- wave 1: NEW FIELD supplier (2025-01-10) ---------------------------------
            (1,  "Laptop Pro",  "Computers",   "Acer",       "Graphite", "TechSource",   None, "update", "2025-01-10"),
            (3,  "Coffee Mug",  "Home",        "Bodum",      "Ivory",    "HomeGoods Co", None, "update", "2025-01-10"),
            (8,  "Smart Watch", "Electronics", "Apple",      "Black",    "AppleDist",    None, "update", "2025-01-10"),
            (30, "E-Reader",    "Electronics", "Kobo",       "Black",    "TechSource",   None, "insert", "2025-01-10"),
            # --- wave 2: NEW FIELD warranty_months (2025-02-15+) -------------------------
            (2,  "Headphones",  "Audio",       "Sennheiser", "Navy",     "AudioPlus",    24,   "update", "2025-02-15"),
            (30, "E-Reader",    "Electronics", "Kobo",       "Sand",     "BookDist",     12,   "update", "2025-02-15"),
            (1,  "Laptop Pro",  "Computers",   "Acer",       "Graphite", "TechSource",   36,   "update", "2025-02-20"),
        ],
    },
}

def generate_evolution_feed(fields=(("data_owner", "string"), ("quality_score", "bigint"))):
    """Same shape as the curated feed, for any dimension: 4 active entities + 1 new one."""
    active = [tuple(r) for r in (dim_t9_final
              .filter(F.expr("element_at(_scd, -1).valid_to") == F.lit(date(9999, 12, 31)))
              .select(*cols()).orderBy(KEY).limit(4).collect())]
    assert len(active) == 4, "the schema-evolution demo needs at least 4 active entities"
    a, b, c, d = active
    k_new = new_key(1)
    events = [
        (*a, "owner_1", None, "update", "2025-01-10"),
        (*b, "owner_2", None, "update", "2025-01-10"),
        (*c, "owner_3", None, "update", "2025-01-10"),
        (k_new, *a[1:], "owner_1", None, "insert", "2025-01-10"),
        (*d, "owner_2", 24, "update", "2025-02-15"),
        (k_new, *a[1:], "owner_4", 12, "update", "2025-02-15"),
        (*a, "owner_1", 36, "update", "2025-02-20"),
    ]
    return {"fields": list(fields), "events": events}

EVOLUTION = SAMPLE_EVOLUTION_FEEDS.get(DIM_NAME) if IS_SAMPLE else None
EVOLUTION = EVOLUTION or generate_evolution_feed()
EVO_FIELDS = EVOLUTION["fields"]                    # [(name, type), ...]
NEW_FIELDS = [f for f, _ in EVO_FIELDS]
EVO_ATTRS  = ATTRS + NEW_FIELDS                     # attributes after the evolution

EVO_DDL = ", ".join([f"{q(f.name)} {f.dataType.simpleString()}" for f in BASE_SCHEMA.fields]
                    + [f"{q(f)} {t}" for f, t in EVO_FIELDS]
                    + ["change_type string", "change_date string"])
schema_evolution_events = spark.createDataFrame(EVOLUTION["events"], EVO_DDL)
EVO_KEYS = list(dict.fromkeys(e[0] for e in EVOLUTION["events"]))   # touched entities, in feed order

display(schema_evolution_events.orderBy("change_date", KEY))

### Type 9 + schema evolution — applying the new-fields feed

1. The existing `_scd` `ARRAY<STRUCT>` is widened in place: every existing version element gains supplier / `warranty_months` as NULL — those fields did not exist at that version's time (elasticity).
2. The feed creates new version elements that carry the new fields.
3. The previous last version of each touched product is closed (`valid_to` = next `change_date` - 1), versions keep numbering 0,1,2,…
4. The regular (current) columns also gain the new attributes — the table itself evolves schema, no backfill, no flag column.


In [0]:
print("_scd struct BEFORE schema evolution (no " + " / ".join(NEW_FIELDS) + "):")
dim_t9_final.printSchema()

# --- Step 1: widen every existing _scd element with the new fields ----
t9_base = dim_t9_final.withColumn(
    "_scd",
    F.transform(
        F.col("_scd"),
        lambda e: F.struct(
            *[e[c].alias(c) for c in SCD_META + ATTRS],
            *[F.lit(None).cast(t).alias(f) for f, t in EVO_FIELDS],
        ),
    ),
)

baseline_versions = (
    t9_base.select(KEY, F.explode("_scd").alias("v"))
    .select(KEY, "v.*")
)

# --- Step 2: new versions from the evolving feed ----------------------
w_new = Window.partitionBy(KEY).orderBy("change_date")

# Guard: an event is applied only if (a) the entity is NEW (not in the baseline),
# or (b) its last baseline version is still OPEN and the event is dated after
# that version started. Events for entities that were already deleted would
# silently reopen a closed entity (and a replayed feed would duplicate
# versions), so they are skipped and counted instead.
open_base = (
    baseline_versions
    .filter(F.col("valid_to") == F.to_date(F.lit("9999-12-31")))
    .select(KEY, F.col("valid_from").alias("open_from"))
)
known_base = baseline_versions.select(KEY).distinct().withColumn("is_known", F.lit(True))

events_in = (
    schema_evolution_events
    .filter(F.col("change_type") != "delete")
    .withColumn("change_date", F.to_date("change_date"))
)
events_ok = (
    events_in
    .join(known_base, KEY, "left")
    .join(open_base, KEY, "left")
    .filter(F.col("is_known").isNull()
            | (F.col("open_from").isNotNull() & (F.col("change_date") > F.col("open_from"))))
    .drop("is_known", "open_from")
)
print(f"Feed events: {events_in.count()}, applied: {events_ok.count()}, "
      f"skipped by the guard (deleted entity / replay): {events_in.count() - events_ok.count()}")

new_versions = (
    events_ok
    .withColumn("rn_new", F.row_number().over(w_new))
    .withColumn("next_date", F.lead("change_date").over(w_new))
    .withColumn("valid_from", F.col("change_date"))
    .withColumn("valid_to", F.coalesce(F.date_sub(F.col("next_date"), 1), F.to_date(F.lit("9999-12-31"))))
    .join(t9_base.select(KEY, F.size("_scd").alias("n_base")), KEY, "left")
    .withColumn("version", (F.coalesce(F.col("n_base"), F.lit(0)) + F.col("rn_new") - 1).cast("long"))
)

# --- Step 3: close the previous last version where new events arrive --
first_new_date = (
    new_versions.groupBy(KEY)
    .agg(F.min("valid_from").alias("first_new_date"))
)
baseline_closed = (
    baseline_versions
    .join(first_new_date, KEY, "left")
    .withColumn(
        "valid_to",
        F.when(
            F.col("first_new_date").isNotNull() & (F.col("valid_to") == F.to_date(F.lit("9999-12-31"))),
            F.date_sub(F.col("first_new_date"), 1),
        ).otherwise(F.col("valid_to")),
    )
)

version_cols = [KEY, *SCD_META, *EVO_ATTRS]
all_versions = baseline_closed.select(*version_cols).unionByName(new_versions.select(*version_cols))

# --- Step 4: re-assemble _scd arrays and the evolved current columns --
scd_arrays = all_versions.groupBy(KEY).agg(
    F.sort_array(F.collect_list(F.struct(*SCD_META, *EVO_ATTRS))).alias("_scd")
)

w_latest = Window.partitionBy(KEY).orderBy(F.col("version").desc())
current = (
    all_versions
    .withColumn("rn", F.row_number().over(w_latest))
    .filter(F.col("rn") == 1)
    .select(KEY, *EVO_ATTRS)
)

dim_t9_evolved = current.join(scd_arrays, KEY).orderBy(KEY)

print("\n_scd struct AFTER schema evolution — " + " and ".join(NEW_FIELDS)
      + " are now part of every version element:")
dim_t9_evolved.printSchema()

# Current snapshot — deleted entities are recognised via the LAST _scd version's
# valid_to — no flag column, consistent with the Type 9 design rule.
last_valid_to = F.expr("element_at(_scd, size(_scd)).valid_to")
print("\nCurrent snapshot (active entities) — the new attributes are plain columns now (the table itself evolved schema):")
display(
    dim_t9_evolved
    .filter(last_valid_to == F.to_date(F.lit("9999-12-31")))
    .drop("_scd")
    .orderBy(KEY)
)

### Persist Type 9 + schema evolution to A real Delta table

1. Baseline: the Type 9 table as it was before the new fields existed.
2. `ALTER TABLE` widens the table and the `_scd` element struct in place (metadata-only change: existing data files are not rewritten).
3. MERGE writes only the rows whose `_scd` array grew (plus new entities).

Every step is a Delta commit, so the table history (chapter 13) records the evolution as well. Change Data Feed is enabled so row-level changes can be compared with the `_scd` array later.


In [0]:
TBL = TBL_T9_EVOLUTION   # from the parameters cell (section 0)

(dim_t9_final.write.format("delta").mode("overwrite")
    .option("overwriteSchema", "true").saveAsTable(TBL))

spark.sql(f"ALTER TABLE {TBL} SET TBLPROPERTIES (delta.enableChangeDataFeed = true)")
spark.sql(f"ALTER TABLE {TBL} ADD COLUMNS ({', '.join(f'{q(f)} {t.upper()}' for f, t in EVO_FIELDS)})")
spark.sql(f"ALTER TABLE {TBL} ADD COLUMNS ({', '.join(f'_scd.element.{q(f)} {t.upper()}' for f, t in EVO_FIELDS)})")

dim_t9_evolved.createOrReplaceTempView("t9_evolved_src")
spark.sql(f"""
MERGE INTO {TBL} AS t
USING t9_evolved_src AS s
  ON {sql_on([KEY], 't', 's')}
WHEN MATCHED AND size(s._scd) <> size(t._scd) THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *
""")

spark.table(TBL).printSchema()
display(spark.sql(f"DESCRIBE HISTORY {TBL}").select("version", "timestamp", "operation", "operationMetrics").orderBy("version"))

### How `_scd` behaves and evolves — version-by-version view (from Delta)

Read the persisted Delta table back and explode the evolved `_scd` for the products touched by the new feed. All elements of the array share one struct type: versions created before the new fields existed read them as NULL (field not known at that time), versions created after carry values.


In [0]:
t9_evo = spark.table(TBL)

print("History of the entities touched by the schema-evolution feed:")
display(
    t9_evo
    .filter(F.col(KEY).isin(EVO_KEYS))
    .select(KEY, F.explode("_scd").alias("v"))
    .select(KEY, "v.*")
    .orderBy(KEY, "version")
)

print(f"{KEY} = {EVO_KEYS[0]} — the raw _scd array: versions built before the feed read "
      + " / ".join(NEW_FIELDS) + " as NULL, later versions carry them:")
display(t9_evo.filter(F.col(KEY) == EVO_KEYS[0]).select(KEY, "_scd"))

# 13. Delta table history of the SCD tables (types 0–9) vs the `_scd` array

Two different histories exist side by side and are easy to confuse.

| | Delta table history (`DESCRIBE HISTORY`, time travel, Change Data Feed) | `_scd` array (Type 9) |
|---|---|---|
| Time axis | **System time**: when the platform committed the change | **Business (valid) time**: when the value was true in the real world (`valid_from` / `valid_to`) |
| Granularity | One entry per commit (a batch of changes) | One element per entity version |
| Written by | Delta, automatically, for every table of every SCD type | The load logic (MERGE / PySpark), on purpose |
| Retention | Limited by log retention (default 30 days) and by `VACUUM` (default 7 days for data files) | As long as the row exists |
| Late-arriving data | Shows *when* it was loaded | Shows *where* it belongs in business time |
| Typical question | "What did the table look like at 10:00 yesterday, and what changed it?" | "What did the dimension look like on 2024-09-15?" |

Sections: **13.A** commit history of all SCD tables (0–9), **13.B** reconciling Delta versions with `_scd` elements, **13.C** example queries that put both side by side, **13.D** retention.

Note: a table that was re-created and re-loaded many times in this notebook keeps all those commits in its history, so commit counts reflect how often the demo was run, not how the SCD type behaves.

### 13.A Delta commit history of the SCD tables (Types 0-9)

Type 0 had no table in section 8 (no MERGE example), so it is created here: insert-only MERGE, updates and deletes are ignored.


In [0]:
spark.sql(f"CREATE OR REPLACE TABLE {TBL_T0} AS SELECT {sql_cols(cols())} FROM dim_base_v")
spark.sql(f"""
MERGE INTO {TBL_T0} AS tgt
USING (
    SELECT {sql_cols(cols())} FROM (
        SELECT *, ROW_NUMBER() OVER (PARTITION BY {K} ORDER BY change_date) AS rn
        FROM changes_v WHERE change_type = 'insert'
    ) WHERE rn = 1
) AS src
  ON {sql_on([KEY])}
WHEN NOT MATCHED THEN INSERT ({sql_cols(cols())}) VALUES ({sql_cols(cols(), 'src.')})
""")

SCD_TABLES = {
    "Type 0": [TBL_T0],
    "Type 1": [TBL_T1],
    "Type 2": [TBL_T2],
    "Type 3": [TBL_T3],
    "Type 4": [TBL_T4_BASE, TBL_T4_MINI],
    "Type 5": [TBL_T5_BASE, TBL_T5_MINI],
    "Type 6": [TBL_T6, TBL_T6_CURRENT],
    "Type 7": [TBL_T7_CURRENT, TBL_T7_HISTORY],
    "Type 8": [TBL_T8],
    "Type 9": [TBL_T9, TBL_T9_EVOLUTION],
}

def history_of(scd_type, full_name):
    m = F.col("operationMetrics")
    return spark.sql(f"DESCRIBE HISTORY {full_name}").select(
        F.lit(scd_type).alias("scd_type"),
        F.lit(full_name.split(".")[-1]).alias("table"),
        "version", "timestamp", "operation",
        m["numTargetRowsInserted"].cast("long").alias("rows_inserted"),
        m["numTargetRowsUpdated"].cast("long").alias("rows_updated"),
        m["numTargetRowsDeleted"].cast("long").alias("rows_deleted"),
        m["numOutputRows"].cast("long").alias("rows_written"),
    )

delta_history = reduce(
    DataFrame.unionByName,
    [history_of(t, n) for t, names in SCD_TABLES.items() for n in names],
)
delta_history.createOrReplaceTempView("scd_delta_history")

print("Every commit of every SCD table - one row per Delta version:")
display(delta_history.orderBy("scd_type", "table", "version"))

print("Per table: number of commits and what they did:")
display(
    delta_history.groupBy("scd_type", "table")
    .agg(F.count("*").alias("commits"),
         F.collect_list("operation").alias("operations"),
         F.sum("rows_inserted").alias("rows_inserted"),
         F.sum("rows_updated").alias("rows_updated"),
         F.sum("rows_deleted").alias("rows_deleted"))
    .orderBy("scd_type", "table")
)

### 13.B reconcile the two histories: Delta versions vs `_scd` elements

Delta time travel shows the table as it was after each commit (system time). The `_scd` array shows every version of every entity (business time). The feed loaded in one MERGE contained 7 events, so the number of `_scd` elements must grow by exactly 7 between the Delta version before the MERGE and the version produced by it (both looked up below, not hard-coded).


In [0]:
# The schema-evolution MERGE is the latest MERGE commit of the table; the
# version just before it is the "before" snapshot. Reused by 13.C1-13.C3.
V_AFTER  = (spark.sql(f"DESCRIBE HISTORY {TBL}").where("operation = 'MERGE'")
            .agg(F.max("version")).first()[0])
V_BEFORE = V_AFTER - 1
print(f"Delta versions used below: before = {V_BEFORE}, after (MERGE) = {V_AFTER}")

schema_evolution_events.createOrReplaceTempView("schema_evolution_events_v")

display(spark.sql(f"""
SELECT 'Delta VERSION AS OF {V_BEFORE} (before the MERGE)' AS snapshot,
       count(*) AS entities, sum(size(_scd)) AS scd_elements
FROM {TBL} VERSION AS OF {V_BEFORE}
UNION ALL
SELECT 'Delta VERSION AS OF {V_AFTER} (after the MERGE)', count(*), sum(size(_scd))
FROM {TBL} VERSION AS OF {V_AFTER}
UNION ALL
SELECT 'Events in the schema-evolution feed', count(DISTINCT {K}), count(*)
FROM schema_evolution_events_v
"""))

### 13.C1 row-level changes from the Delta log (Change Data Feed)

CDF was enabled at Delta version 1, the MERGE is version `V_AFTER` (see 13.B). Every changed row appears as `update_preimage` / `update_postimage` / insert with the commit that wrote it. Compare size(`_scd`) before and after: it grows by the number of events the feed carried for that product.


In [0]:
display(spark.sql(f"""
SELECT {K}, _change_type, _commit_version, _commit_timestamp,
       size(_scd) AS scd_elements, {sql_cols(NEW_FIELDS)}
FROM table_changes('{TBL}', {V_AFTER})
ORDER BY {K}, _commit_version, _change_type
"""))

### 13.C2 two time axes: when the change was true vs when it was loaded

The `_scd` elements added by the MERGE carry business dates (`valid_from`, 2025), the Delta commit that wrote them carries system time (today). The same row change, seen on two different axes.


In [0]:
display(spark.sql(f"""
WITH ch AS (SELECT * FROM table_changes('{TBL}', {V_AFTER})),
post AS (SELECT {K}, _commit_version, _commit_timestamp, _scd FROM ch WHERE _change_type = 'update_postimage'),
pre  AS (SELECT {K}, size(_scd) AS n_before FROM ch WHERE _change_type = 'update_preimage'),
new_elements AS (
  SELECT p.{K}, p._commit_version, p._commit_timestamp,
         explode(filter(p._scd, e -> e.version >= pre.n_before)) AS v
  FROM post p JOIN pre ON p.{K} = pre.{K}
)
SELECT {K}, v.version, v.valid_from AS business_valid_from, v.valid_to AS business_valid_to,
       {sql_cols(NEW_FIELDS, 'v.')}, _commit_version, _commit_timestamp AS system_commit_time
FROM new_elements
ORDER BY {K}, v.version
"""))

### 13.C3 as-of questions: business time (`_scd`) vs system time (Delta)

(a) What was true for product 1 on 2025-01-15? → `_scd` of the current table.
(b) What did the table say about product 1 before the feed was loaded? → Delta time travel (version `V_BEFORE`), last `_scd` element.
(a) can be answered forever; (b) only while that Delta version is retained.


In [0]:
K0, F0 = sql_lit(EVO_KEYS[0]), q(NEW_FIELDS[0])     # first entity of the feed, first new field

display(spark.sql(f"""
SELECT 'a) business as-of 2025-01-15 (_scd, current table)' AS question,
       {K}, v.version, v.{F0}, v.valid_from, v.valid_to
FROM (SELECT {K}, explode(_scd) AS v FROM {TBL})
WHERE {K} = {K0} AND v.valid_from <= DATE'2025-01-15' AND v.valid_to >= DATE'2025-01-15'
UNION ALL
SELECT 'b) system as-of Delta version {V_BEFORE} (before the load): last _scd element',
       {K}, element_at(_scd, -1).version, element_at(_scd, -1).{F0},
       element_at(_scd, -1).valid_from, element_at(_scd, -1).valid_to
FROM {TBL} VERSION AS OF {V_BEFORE}
WHERE {K} = {K0}
"""))

### 13.D retention: how far back can each history be read?

Delta history only starts when the table was first written here and is limited by log retention / `VACUUM`. `_scd` reaches back as far as the business data does, independent of when the rows were loaded.


In [0]:
oldest_delta = spark.sql(f"DESCRIBE HISTORY {TBL}").agg(F.min("timestamp").alias("oldest_delta_commit"))
oldest_scd = (spark.table(TBL).select(F.explode("_scd").alias("v"))
              .agg(F.min("v.valid_from").alias("oldest_scd_valid_from")))
display(oldest_delta.crossJoin(oldest_scd))

display(spark.sql(f"SHOW TBLPROPERTIES {TBL}"))

## 13.E What each history can and cannot answer

**Delta history / time travel / Change Data Feed** answers: when was the table changed, by which operation, how many rows were inserted, updated or deleted, and what the table looked like after any retained commit. It works the same for every SCD type, because it records what happened to the *table*. It cannot tell you when a change was true in the real world, and it cannot see anything that happened before the table was first written here.

**The `_scd` array** answers: what was true on any business date, in which order the versions occurred, and what a late-arriving change did to the timeline. It is part of the data, so it survives `VACUUM`, log expiry and table re-creation. It cannot tell you when a version was loaded, by which job, or what the table said before a correction, unless you add such fields to the struct (for example `record_source` or a load timestamp).

**Used together** they give a bi-temporal view: `_scd` for business time, the Delta log for system time. A practical audit pattern is to keep the Change Data Feed for the retention window you need and to join its commit metadata to the `_scd` elements that each commit added (see 13.C2).

Types 0–7 keep their history either nowhere (0, 1), in rows (2, 6, 7), in columns (3) or in a second table (4, 5, 7). Type 8 (fact-date rows) keeps it in fact-aligned rows — one row per date, never the whole timeline on one row. In all of Types 0–8 the Delta log is the only record of *when* the rows were written; only Type 9 puts the full business timeline into the same row that the Delta log versions.

# 14. Stress test — bulk load of a randomly-sized change feed

Real feeds are not 7 rows. This chapter loads a **random number of incoming
rows (between 1 and 1.5 million)** into the SCD Type 9 dimension — the
pattern that combines every load concern at once — and measures:

* **how long** each phase takes (feed generation, table seed, MERGE loads,
  schema evolution),
* **what resources** the load consumed (rows inserted/updated, bytes and
  files added/removed, deletion vectors, and the scan / rewrite / source-
  materialisation time split — straight from Delta's own commit metrics),
* and anything remarkable **spotted during the transformation**.

The feed carries **all change kinds at once**, split into three waves
(mimicking three source batches):

1. `2025-03-01` — bulk **inserts** of brand-new products (ids ≥ 10,000,000)
2. `2025-04-01` — **updates** of a sample of wave-1 products plus the original 30 (each appended version grows the `_scd` array)
3. `2025-05-01` — **deletes** of a sample (last `_scd` version closed, no flag column) plus **updates carrying a brand-new column** — `stock_qty` — so **schema evolution** happens mid-load, after two waves were already loaded on the old schema

The target is a fresh copy of the Type 9 evolution table
(`scd_t9_stress`, name from the parameters cell), so the run is repeatable
and never touches the demo tables. Timing is wall-clock
(`time.perf_counter`); resource usage comes from Delta's commit metrics
(`DESCRIBE HISTORY` → `operationMetrics`) and `DESCRIBE DETAIL`.

### 14.A stress feed — random size, all change kinds, new column

`N_ROWS` is drawn pseudo-randomly (fixed seed) between 1 and 1.5 million, so every rerun is reproducible; change seed to draw another size. It is split into three waves that a real source might send as three batches: wave 1 (2025-03-01): bulk inserts of brand-new products wave 2 (2025-04-01): update events (multi-version `_scd` arrays grow) wave 3 (2025-05-01): deletes + updates that carry a brand-new column (`stock_qty`) — schema evolution arrives mid-load The feed is cached + counted so that (a) the random sampling cannot be recomputed differently by the MERGEs, and (b) generation is a measurable phase of its own.


In [0]:
SEED = 42                                              # fixed seed: reruns draw the same size
random.seed(SEED)
N_ROWS = random.randint(1_000_000, 1_500_000)          # random load size (reproducible)
N_W1_INSERTS = int(N_ROWS * 0.60)
N_W2_UPDATES = int(N_ROWS * 0.25)
N_W3_DELETES = int(N_ROWS * 0.08)
N_W3_UPDATES = N_ROWS - N_W1_INSERTS - N_W2_UPDATES - N_W3_DELETES

print(f"Randomly drawn stress-feed size: {N_ROWS:,} rows")
print(f"  wave 1 (inserts):               {N_W1_INSERTS:,}")
print(f"  wave 2 (updates):               {N_W2_UPDATES:,}")
print(f"  wave 3 (deletes):               {N_W3_DELETES:,}")
print(f"  wave 3 (updates w/ new column): {N_W3_UPDATES:,}")

# --- the column model of the target (the Type 9 evolution table of section 12) ------
EVO_TABLE      = spark.table(TBL_T9_EVOLUTION)
STRESS_ATTRS   = business_cols(EVO_TABLE)                   # attributes incl. the section-12 fields
STRESS_NEW_COL, STRESS_NEW_TYPE = SAMPLE.get("stress_new_col", ("load_batch", "int"))
STRESS_KEY_OFFSET = 1_000                                   # new_key(0..999) is reserved for the demos
ORIG_KEYS = [r[0] for r in dim_t0.select(KEY).orderBy(KEY).collect()]   # every entity of section 4

# realistic values: each attribute picks from (up to 20) values it already takes in the table
VALUE_POOLS = {a: [r[0] for r in EVO_TABLE.where(F.col(a).isNotNull()).select(a).distinct()
                                         .orderBy(a).limit(20).collect()]
               for a in STRESS_ATTRS}

def pick(values, *key_cols):
    """Deterministic pseudo-random pick from a literal list (1-based)."""
    return F.element_at(
        F.array(*[F.lit(v) for v in values]),
        F.pmod(F.hash(*key_cols), F.lit(len(values))) + F.lit(1),
    )

def attr_values(seed_col):
    """Every stress attribute, picked from its own pool (typed NULL when the pool is empty)."""
    return [(pick(VALUE_POOLS[a], seed_col, F.lit(a)) if VALUE_POOLS[a] else F.lit(None))
            .cast(col_type(a, EVO_TABLE)).alias(a) for a in STRESS_ATTRS]

def null_attrs():
    return [F.lit(None).cast(col_type(a, EVO_TABLE)).alias(a) for a in STRESS_ATTRS]

def wave1_key(n_col):
    """Key of the n-th wave-1 entity (brand-new, far above the demo keys)."""
    return new_key_col(F.lit(STRESS_KEY_OFFSET) + n_col)

def one_event_per_key(df):
    """MERGE allows at most one source row per target row — keep one per key."""
    w = Window.partitionBy(KEY).orderBy("_rid")
    return (df.withColumn("rn", F.row_number().over(w))
              .filter(F.col("rn") == 1).drop("rn", "_rid"))

t_gen = time.perf_counter()

# --- wave 1: bulk INSERTS of brand-new entities ------------------------------------
w1 = (
    spark.range(0, N_W1_INSERTS, numPartitions=64)
    .select(
        F.col("id").alias("_rid"),
        wave1_key(F.col("id")).alias(KEY),
        *attr_values(F.col("id")),
        F.lit(None).cast(STRESS_NEW_TYPE).alias(STRESS_NEW_COL),   # column does not exist yet -> NULL
        F.lit("insert").alias("change_type"),
        F.to_date(F.lit("2025-03-01")).alias("change_date"),
    )
    .drop("_rid")
)

# --- wave 2: UPDATEs — 20% hit the original entities, 80% hit wave-1 entities -------
w2 = one_event_per_key(
    spark.range(0, N_W2_UPDATES, numPartitions=64)
    .select(
        F.col("id").alias("_rid"),
        F.when(F.rand() < 0.2, pick(ORIG_KEYS, "id"))
         .otherwise(wave1_key((F.rand() * F.lit(N_W1_INSERTS)).cast("long")))
         .cast(KEY_TYPE).alias(KEY),
        *attr_values(F.col("id") + F.lit(1)),
        F.lit(None).cast(STRESS_NEW_TYPE).alias(STRESS_NEW_COL),
        F.lit("update").alias("change_type"),
        F.to_date(F.lit("2025-04-01")).alias("change_date"),
    )
)

# --- wave 3a: DELETEs (wave-1 entities only; attributes are irrelevant) -------------
w3_del = one_event_per_key(
    spark.range(0, N_W3_DELETES, numPartitions=32)
    .select(
        F.col("id").alias("_rid"),
        wave1_key((F.rand() * F.lit(N_W1_INSERTS)).cast("long")).alias(KEY),
        *null_attrs(),
        F.lit(None).cast(STRESS_NEW_TYPE).alias(STRESS_NEW_COL),
        F.lit("delete").alias("change_type"),
        F.to_date(F.lit("2025-05-01")).alias("change_date"),
    )
)

# --- wave 3b: UPDATEs carrying the NEW COLUMN ---------------------------------------
w3_upd = one_event_per_key(
    spark.range(0, N_W3_UPDATES, numPartitions=64)
    .select(
        F.col("id").alias("_rid"),
        F.when(F.rand() < 0.2, pick(ORIG_KEYS, "id"))
         .otherwise(wave1_key((F.rand() * F.lit(N_W1_INSERTS)).cast("long")))
         .cast(KEY_TYPE).alias(KEY),
        *attr_values(F.col("id") + F.lit(2)),
        (F.pmod(F.hash("id"), F.lit(60)) + F.lit(1)).cast(STRESS_NEW_TYPE).alias(STRESS_NEW_COL),  # NEW COLUMN's data
        F.lit("update").alias("change_type"),
        F.to_date(F.lit("2025-05-01")).alias("change_date"),
    )
)
# one event per entity per wave: drop update events for deleted entities
w3_upd = w3_upd.join(w3_del.select(KEY), KEY, "left_anti")

# --- materialise (freeze the sampling) + measure generation ------------------------
w1.cache();     W1_COUNT     = w1.count()
w2.cache();     W2_COUNT     = w2.count()
w3_del.cache(); W3_DEL_COUNT = w3_del.count()
w3_upd.cache(); W3_UPD_COUNT = w3_upd.count()
W3_COUNT = W3_DEL_COUNT + W3_UPD_COUNT
GEN_TIME = time.perf_counter() - t_gen

w1.createOrReplaceTempView("stress_w1")
w2.createOrReplaceTempView("stress_w2")
w3_del.unionByName(w3_upd).createOrReplaceTempView("stress_w3")

print(f"\nEvents after one-event-per-key dedup:")
print(f"  stress_w1 (inserts):                       {W1_COUNT:,}")
print(f"  stress_w2 (updates):                       {W2_COUNT:,}")
print(f"  stress_w3 (deletes {W3_DEL_COUNT:,} + updates {W3_UPD_COUNT:,}): {W3_COUNT:,}")
print(f"Feed generation time (incl. materialisation): {GEN_TIME:.1f} s")

print(f"\nSample of wave 1 (bulk inserts, {STRESS_NEW_COL} still NULL):")
display(w1.orderBy(KEY).limit(5))
print(f"Sample of wave 3 updates (carry the new {STRESS_NEW_COL} column):")
display(w3_upd.orderBy(KEY).limit(5))

### 14.B load & measure — seed, three MERGE waves, schema evolution

Target: a fresh copy of the Type 9 evolution table (repeatable run). One Type 9 incremental MERGE per wave (at most one event per product, as MERGE requires). Wave 3 arrives with a new column → two metadata-only `ALTER TABLE`s widen the table and the `_scd` struct first.


In [0]:
timings = {}

def timed(label, fn):
    t0 = time.perf_counter()
    out = fn()
    timings[label] = time.perf_counter() - t0
    print(f"{label}: {timings[label]:.1f} s")
    return out

# --- phase 0: seed the stress table from the evolution table -----------------------
def _seed():
    spark.sql(f"CREATE OR REPLACE TABLE {TBL_T9_STRESS} AS SELECT * FROM {TBL_T9_EVOLUTION}")
timed("seed — fresh copy of the Type 9 evolution table", _seed)

baseline = spark.table(TBL_T9_STRESS).agg(
    F.count("*").alias("rows"),
    F.sum(F.size("_scd")).alias("scd_elements"),
).collect()[0]
ROWS_BEFORE, ELEMENTS_BEFORE = baseline["rows"], baseline["scd_elements"]
print(f"baseline: {ROWS_BEFORE} rows / {ELEMENTS_BEFORE} _scd elements")

# --- the Type 9 incremental MERGE, generated for the CURRENT schema ---------------
OPEN_DATE = "DATE'9999-12-31'"

def type9_merge_sql(view, attrs):
    """Type 9 incremental MERGE: update = close last version + append a new element to _scd +
    overwrite current columns; delete = close the last version only (no flag column);
    insert = new row with _scd = [version 0]."""
    is_open_last = f"e.version = size(tgt._scd) - 1 AND e.valid_to = {OPEN_DATE}"
    e_closed = (f"named_struct('version', e.version, 'valid_from', e.valid_from, "
                f"'valid_to', date_sub(src.change_date, 1), {sql_struct(attrs, 'e.')})")
    e_new    = (f"named_struct('version', CAST(size(tgt._scd) AS BIGINT), "
                f"'valid_from', src.change_date, 'valid_to', {OPEN_DATE}, {sql_struct(attrs, 'src.')})")
    e_zero   = (f"named_struct('version', CAST(0 AS BIGINT), "
                f"'valid_from', src.change_date, 'valid_to', {OPEN_DATE}, {sql_struct(attrs, 'src.')})")
    return f"""
MERGE INTO {TBL_T9_STRESS} AS tgt
USING (
    -- Revival rule (same as the guard in section 12): events for an entity whose
    -- last _scd version is already CLOSED (deleted) are skipped, never re-opened
    SELECT s.* FROM {view} s
    LEFT ANTI JOIN (SELECT {K} FROM {TBL_T9_STRESS}
                    WHERE element_at(_scd, -1).valid_to <> {OPEN_DATE}) closed
      ON s.{K} = closed.{K}
) AS src
  ON {sql_on([KEY])}
WHEN MATCHED AND src.change_type = 'delete' THEN UPDATE SET
    tgt._scd = transform(tgt._scd, e -> IF({is_open_last}, {e_closed}, e))
WHEN MATCHED AND src.change_type = 'update' THEN UPDATE SET
    {sql_set(attrs)},
    tgt._scd = concat(transform(tgt._scd, e -> IF({is_open_last}, {e_closed}, e)), array({e_new}))
WHEN NOT MATCHED AND src.change_type != 'delete' THEN
  INSERT ({sql_cols([KEY, *attrs, '_scd'])})
  VALUES ({sql_cols([KEY, *attrs], 'src.')}, array({e_zero}))
"""

# --- revival rule bookkeeping: events skipped because the entity is already closed
SKIPPED = {}

def count_skipped(view):
    rows = spark.sql(f"""
        SELECT s.change_type, count(*) AS n
        FROM {view} s JOIN {TBL_T9_STRESS} t ON s.{K} = t.{K}
        WHERE element_at(t._scd, -1).valid_to <> {OPEN_DATE}
        GROUP BY s.change_type""").collect()
    for r in rows:
        SKIPPED[r["change_type"]] = SKIPPED.get(r["change_type"], 0) + r["n"]
        print(f"{view}: {r['n']:,} {r['change_type']} event(s) skipped — entity already deleted")

# --- phase 1: wave 1 — bulk inserts ------------------------------------------------
count_skipped("stress_w1")
timed("wave 1 MERGE — bulk inserts", lambda: spark.sql(type9_merge_sql("stress_w1", STRESS_ATTRS)))

# --- phase 2: wave 2 — updates (existing rows, _scd arrays grow) -------------------
count_skipped("stress_w2")
timed("wave 2 MERGE — updates", lambda: spark.sql(type9_merge_sql("stress_w2", STRESS_ATTRS)))

# --- phase 3: schema evolution arrives (metadata-only, no data rewrite) ------------
def _evolve():
    spark.sql(f"ALTER TABLE {TBL_T9_STRESS} ADD COLUMNS ({q(STRESS_NEW_COL)} {STRESS_NEW_TYPE.upper()})")
    spark.sql(f"ALTER TABLE {TBL_T9_STRESS} ADD COLUMNS (_scd.element.{q(STRESS_NEW_COL)} {STRESS_NEW_TYPE.upper()})")
timed("schema evolution — 2x ALTER TABLE (metadata only)", _evolve)

# --- phase 4: wave 3 — deletes + updates carrying the new column -------------------
count_skipped("stress_w3")
timed("wave 3 MERGE — deletes + updates w/ new column",
      lambda: spark.sql(type9_merge_sql("stress_w3", STRESS_ATTRS + [STRESS_NEW_COL])))

LOAD_TOTAL  = sum(v for k, v in timings.items() if "MERGE" in k)
MERGED_ROWS = W1_COUNT + W2_COUNT + W3_COUNT - sum(SKIPPED.values())   # events the MERGEs applied
print(f"\nTotal MERGE load time: {LOAD_TOTAL:.1f} s")
print(f"  {MERGED_ROWS / LOAD_TOTAL:,.0f} merged events/s   ({MERGED_ROWS:,} events applied after dedup + revival rule)")
print(f"  {N_ROWS / LOAD_TOTAL:,.0f} generated rows/s  ({N_ROWS:,} rows drawn before dedup)")

### 14.C resources used — Delta commit metrics + physical layout

Delta records per-commit operationMetrics: the three MERGE commits tell us rows inserted/updated/copied, bytes and files added/removed, deletion vectors, and the time split (scan / rewrite / source materialisation). describe detail adds the resulting physical layout. The stress table is re-created by every run of 14.B, so its history keeps the commits of earlier runs; only the latest three MERGE commits are shown.


In [0]:
m = F.col("operationMetrics")

print("Per-MERGE commit metrics (from DESCRIBE HISTORY -> operationMetrics):")
display(
    spark.sql(f"DESCRIBE HISTORY {TBL_T9_STRESS}")
    .where(F.col("operation") == "MERGE")
    .orderBy(F.col("version").desc()).limit(3)   # the three MERGE waves of the latest run only
    .select(
        "version",
        F.coalesce(m["numTargetRowsMatchedUpdated"], m["numTargetRowsUpdated"]).cast("long").alias("rows_updated"),
        m["numTargetRowsInserted"].cast("long").alias("rows_inserted"),
        m["numTargetRowsCopied"].cast("long").alias("rows_copied"),
        m["numTargetDeletionVectorsAdded"].cast("long").alias("dvs_added"),
        m["numTargetFilesAdded"].cast("long").alias("files_added"),
        m["numTargetBytesAdded"].cast("long").alias("bytes_added"),
        m["numTargetBytesRemoved"].cast("long").alias("bytes_removed"),
        m["executionTimeMs"].cast("long").alias("exec_ms"),
        m["scanTimeMs"].cast("long").alias("scan_ms"),
        m["rewriteTimeMs"].cast("long").alias("rewrite_ms"),
        m["materializeSourceTimeMs"].cast("long").alias("source_ms"),
    )
    .orderBy("version")
)

detail = spark.sql(f"DESCRIBE DETAIL {TBL_T9_STRESS}").select("numFiles", "sizeInBytes").collect()[0]
print(f"Physical layout after the stress load: {detail['numFiles']} files, "
      f"{detail['sizeInBytes'] / 1e6:,.1f} MB total")

print("\nPopulation shape after the load:")
display(
    spark.table(TBL_T9_STRESS).agg(
        F.count("*").alias("rows_total"),
        F.sum(F.size("_scd")).alias("scd_elements_total"),
        F.max(F.size("_scd")).alias("max_versions_per_entity"),
        F.round(F.avg(F.size("_scd")), 3).alias("avg_versions_per_entity"),
        F.sum(F.when(F.expr("element_at(_scd, size(_scd)).valid_to") < F.to_date(F.lit("9999-12-31")), F.lit(1)).otherwise(F.lit(0))).alias("rows_deleted (closed last version)"),
    )
)

### 14.D verify — expected vs actual + spot check of `_scd` evolution

Expected row growth: baseline + wave-1 inserts (waves 2-3 only target existing products). Expected `_scd` elements: baseline + inserts + updates (deletes close the last version but add no element).


In [0]:
actual = spark.table(TBL_T9_STRESS).agg(
    F.count("*").alias("rows"),
    F.sum(F.size("_scd")).alias("elements"),
).collect()[0]

# events skipped by the revival rule (14.B) add neither rows nor _scd elements
expected_rows      = ROWS_BEFORE + W1_COUNT - SKIPPED.get("insert", 0)
expected_elements  = (ELEMENTS_BEFORE + W1_COUNT + W2_COUNT + W3_UPD_COUNT
                      - SKIPPED.get("insert", 0) - SKIPPED.get("update", 0))

ok_rows      = actual["rows"] == expected_rows
ok_elements  = actual["elements"] == expected_elements
print(f"rows:       expected {expected_rows:,} | actual {actual['rows']:,} | {'OK' if ok_rows else 'MISMATCH'}")
print(f"_scd elems: expected {expected_elements:,} | actual {actual['elements']:,} | {'OK' if ok_elements else 'MISMATCH'}")
assert ok_rows and ok_elements, "stress load did not produce the expected table shape"

print(f"\nSpot check — entities whose LAST version carries the new {STRESS_NEW_COL} column")
print(f"(older versions in the same array legitimately read {STRESS_NEW_COL} as NULL):")
display(
    spark.table(TBL_T9_STRESS)
    .where(F.col(STRESS_NEW_COL).isNotNull() & (F.size("_scd") >= 3))
    .select(KEY, STRESS_ATTRS[0], STRESS_NEW_COL, "_scd")
    .limit(3)
)

print(f"Schema after the stress load ({STRESS_NEW_COL} added top-level AND inside _scd):")
spark.table(TBL_T9_STRESS).printSchema()

### 14.E Findings — what the stress test observed

**Reference run** (2026-11-04, serverless compute, SEED = 42, so the load size is reproducible): 1,335,243 generated rows. After the one-event-per-product dedup that MERGE requires, 1,190,421 events reached the MERGEs (8 of them — updates for demo products that were already deleted — are then skipped by the revival rule, so 1,190,413 are applied) — wave 1: 801,145 inserts · wave 2: 226,841 updates (deduped from 333,810) · wave 3: 162,435 events (100,097 deletes + 62,338 updates carrying the new `stock_qty` column). Wall-clock times vary a little from run to run; cells 14.A–14.D print the figures of the latest run.

**Timings (wall-clock, reference run)**

| Phase | Time | Notes |
|---|---|---|
| Feed generation (incl. materialisation) | ≈ 13 s | `spark.range` + deterministic hash picks is not the bottleneck |
| Seed (copy of the 23-row evolution table) | ≈ 2 s | |
| Wave 1 MERGE — 801,145 inserts | ≈ 5 s | ≈ 160K inserts/s |
| Wave 2 MERGE — 226,841 updates | ≈ 7 s | |
| Schema evolution (2× ALTER TABLE) | ≈ 2 s | metadata-only, no data rewrite |
| Wave 3 MERGE — 162,435 deletes + updates | ≈ 7 s | |
| Total MERGE load | ≈ 19 s | ≈ 63K merged events/s (≈ 71K generated rows/s before dedup) |

**Resources (Delta commit metrics, DESCRIBE HISTORY, latest three MERGE commits)**

| Commit | Rows | Files added | Bytes added | DVs | MERGE execution time |
|---|---|---|---|---|---|
| wave 1 (inserts) | 801,145 inserted | 8 | 4.3 MB | 0 | ≈ 4.3 s |
| wave 2 (updates) | 226,841 updated | 1 | 4.3 MB | 9 | ≈ 5.5 s |
| wave 3 (deletes + updates) | 162,435 updated | 2 | 3.3 MB | 9 | ≈ 5.6 s |

Final table: 801,168 rows, 1,090,384 `_scd` versions (max 9 versions on a single product), in 1 file of 11.0 MB. Expected row and element counts matched exactly (verified in 14.D). What the notebook does **not** measure: CPU, memory or DBUs — only wall-clock time and Delta commit metrics — so nothing below is a claim about how serverless allocated its resources.

**What was spotted during the transformation**

1. **An update costs several times more than an insert, per row.** Wave 2 spent ≈ 5.5 s of MERGE execution on 226,841 updates (≈ 24 µs per update); wave 1 spent ≈ 4.3 s on 801,145 inserts (≈ 5 µs per insert), so roughly 4–5× per row. Every update closes the previous version and rewrites the row's entire `_scd` array.
2. **Deletes are just cheap updates in Type 9** — a delete only closes the last version's `valid_to` (same UPDATE path, no flag column). History is preserved, never destroyed.
3. **Deletion vectors absorbed the updates** — each update wave added only 1–2 data files and 9 DVs, with bytes_removed = 0: changed rows are logically removed via DVs instead of rewriting whole files. No OPTIMIZE was run in this notebook; the table ended as one 11.0 MB file.
4. **Schema evolution mid-load was a non-event** — about 2 s for two metadata-only ALTERs (top-level `stock_qty` and `_scd.element.stock_qty`) between waves 2 and 3; the wave-3 MERGE then handled the new column inside every `named_struct` it writes. Older versions legitimately read `stock_qty` as NULL (see the 14.D spot check).
5. **History is permanent and verifiable** — every applied update event became exactly one new `_scd` element: 1,090,384 = 68 (baseline) + 801,145 + 226,841 + 62,338 − 8 (update events for already-deleted entities, skipped by the revival rule: a deleted entity is never re-opened — the same rule as the guard in section 12 and the Type 2 baseline in 14.F). At ≈ 14 bytes per row (compressed) storage size is not the concern; row width growth as arrays grow for high-churn products is.
6. **No compute tuning was needed** for these volumes on serverless; the feed was cached, so the sampling stayed stable across the MERGEs. This notebook does not measure resource usage, so it makes no claim about autoscaling.

**Takeaway:** at this scale (≈ 1.3M events) the Type 9 pattern ran end to end in well under a minute on serverless in this run: generation and load were each under ≈ 20 s. The incremental MERGE + DV + metadata-only-ALTER combination keeps cost proportional to the changed rows, not to the table size. The pattern's write amplification (whole-array rewrite per update) is visible in the per-row cost and is the first thing that will hurt if change frequency per entity grows.
> 

## 14.F Type 2 baseline — the same stress feed loaded into a classic Type 2 table

Sections 14.A–14.E measured Type 9 on its own. To judge it, the same three waves (`stress_w1`, `stress_w2`, `stress_w3`, same seed, same random feed) are now loaded into a classic **Type 2** table (one row per version, `valid_from`, `valid_to`, `is_current`, surrogate key) on the same serverless compute, and the same questions are asked of both tables. Both tables are created with `CREATE TABLE AS`, so they get the same Delta defaults (including deletion vectors).

- **14.F.1** build and load the Type 2 table, three MERGE waves, compare with Type 9
- **14.F.2** size and commit metrics of both tables
- **14.F.3** reads: current state, point in time, fact join
- **14.F.4** how big can the `_scd` array get, and a late-arriving change

Wall-clock times are measured around each statement (median of three runs for reads). They are indicative for this feed, on this compute, on this day — not a benchmark of Databricks.


In [0]:
# Stress 14.F.1 — Type 2 baseline: same seed table, same three waves, classic Type 2 layout
TBL_T2_STRESS = f"{TBL_T9_STRESS}_type2_baseline"
T2_ATTRS = STRESS_ATTRS
t2_timings = {}

def t2_timed(label, fn):
    t0 = time.perf_counter()
    out = fn()
    t2_timings[label] = time.perf_counter() - t0
    print(f"{label}: {t2_timings[label]:.1f} s")
    return out

# --- phase 0: Type 2 table with a surrogate key, seeded from the same evolution table
def _t2_seed():
    spark.sql(f"""
CREATE OR REPLACE TABLE {TBL_T2_STRESS} (
  {q(SK_COL)} BIGINT GENERATED ALWAYS AS IDENTITY,
  {sql_ddl([KEY, *T2_ATTRS], EVO_TABLE)},
  valid_from DATE,
  valid_to   DATE,
  is_current BOOLEAN)""")
    spark.sql(f"""
INSERT INTO {TBL_T2_STRESS} ({sql_cols([KEY, *T2_ATTRS, *SCD_COLS])})
SELECT d.{K}, {sql_cols(T2_ATTRS, 'e.')}, e.valid_from, e.valid_to, e.valid_to = DATE'9999-12-31'
FROM {TBL_T9_EVOLUTION} d LATERAL VIEW explode(d._scd) x AS e""")
t2_timed("seed — Type 2 table from the evolution table", _t2_seed)

# --- the Type 2 incremental MERGE: every update needs TWO source rows
#     (one closes the current row, one inserts the new version)
def type2_merge_sql(view, attrs):
    columns  = [KEY, *attrs]
    ins_cols = sql_cols(columns + SCD_COLS)
    ins_vals = sql_list(sql_cols(columns, "staged."), "staged.change_date", "DATE'9999-12-31'", "true")
    # Revival rule (same as section 12 and 14.B): events for an entity that has rows
    # but no current row (i.e. it was deleted) are skipped, never re-opened
    src = (f"(SELECT s.* FROM {view} s LEFT ANTI JOIN (SELECT {K} FROM {TBL_T2_STRESS} "
           f"GROUP BY {K} HAVING NOT bool_or(is_current)) closed ON s.{K} = closed.{K})")
    return f"""
MERGE INTO {TBL_T2_STRESS} AS tgt
USING (
  SELECT s.{K} AS merge_key, s.* FROM {src} s
  UNION ALL
  SELECT CAST(NULL AS {KEY_TYPE.simpleString()}) AS merge_key, s.* FROM {src} s
  JOIN {TBL_T2_STRESS} t ON t.{K} = s.{K} AND t.is_current
  WHERE s.change_type = 'update'
) AS staged
ON tgt.{K} = staged.merge_key AND tgt.is_current
WHEN MATCHED AND staged.change_type IN ('update', 'delete') THEN
  UPDATE SET tgt.is_current = false, tgt.valid_to = date_sub(staged.change_date, 1)
WHEN NOT MATCHED AND staged.change_type <> 'delete' THEN
  INSERT ({ins_cols}) VALUES ({ins_vals})
"""

t2_timed("wave 1 MERGE — bulk inserts", lambda: spark.sql(type2_merge_sql("stress_w1", T2_ATTRS)))
t2_timed("wave 2 MERGE — updates",      lambda: spark.sql(type2_merge_sql("stress_w2", T2_ATTRS)))
t2_timed("schema evolution — 1x ALTER TABLE (metadata only)",
         lambda: spark.sql(f"ALTER TABLE {TBL_T2_STRESS} ADD COLUMNS ({q(STRESS_NEW_COL)} {STRESS_NEW_TYPE.upper()})"))
t2_timed("wave 3 MERGE — deletes + updates w/ new column",
         lambda: spark.sql(type2_merge_sql("stress_w3", T2_ATTRS + [STRESS_NEW_COL])))

T2_LOAD_TOTAL = sum(v for k, v in t2_timings.items() if "MERGE" in k)
print(f"\nTotal Type 2 MERGE load time: {T2_LOAD_TOTAL:.1f} s")

# --- verification: Type 2 rows must equal the exploded Type 9 versions, row for row
VCOLS = [KEY, *T2_ATTRS, STRESS_NEW_COL, "valid_from", "valid_to"]
t9_rows = (spark.table(TBL_T9_STRESS).select(KEY, F.explode("_scd").alias("e"))
           .select(KEY, *[F.col(f"e.{c}").alias(c) for c in VCOLS[1:]]))
t2_rows = spark.table(TBL_T2_STRESS).select(*VCOLS)
n_t2 = t2_rows.count()
n_t9 = t9_rows.count()
only_t9 = t9_rows.exceptAll(t2_rows).count()
only_t2 = t2_rows.exceptAll(t9_rows).count()
print(f"Type 2 rows: {n_t2:,} | Type 9 _scd elements: {n_t9:,} | rows only in Type 9: {only_t9} | only in Type 2: {only_t2}")
print("Type 9 loads", f"{LOAD_TOTAL:.1f}", "s vs Type 2", f"{T2_LOAD_TOTAL:.1f}", "s in the three MERGE waves")

In [0]:
# Stress 14.F.2 — size on disk and Delta commit metrics, Type 9 vs Type 2

OPEN = "DATE'9999-12-31'"
tables = [("Type 9", TBL_T9_STRESS), ("Type 2", TBL_T2_STRESS)]

print("Footprint (no OPTIMIZE was run on either table):")
for label, tbl in tables:
    d = spark.sql(f"DESCRIBE DETAIL {tbl}").collect()[0]
    n = spark.table(tbl).count()
    print(f"  {label}: {n:,} rows | {d['numFiles']} file(s) | {d['sizeInBytes'] / 1e6:.1f} MB")

cur9 = spark.sql(f"SELECT count(*) c FROM {TBL_T9_STRESS} WHERE element_at(_scd, -1).valid_to = {OPEN}").collect()[0]["c"]
cur2 = spark.sql(f"SELECT count(*) c FROM {TBL_T2_STRESS} WHERE is_current").collect()[0]["c"]
print(f"Entities that are current: Type 9 {cur9:,} | Type 2 {cur2:,}")

print("\nPer-MERGE commit metrics (DESCRIBE HISTORY -> operationMetrics):")
for label, tbl in tables:
    m = F.col("operationMetrics")
    rows = (spark.sql(f"DESCRIBE HISTORY {tbl}").where(F.col("operation") == "MERGE")
            .orderBy("version")
            .select("version", m["numTargetRowsInserted"].alias("ins"), m["numTargetRowsUpdated"].alias("upd"),
                    m["numTargetRowsCopied"].alias("copied"),
                    m["numTargetDeletionVectorsAdded"].alias("dvs"), m["numTargetFilesAdded"].alias("files"),
                    m["numTargetBytesAdded"].alias("bytes_added"), m["numTargetBytesRemoved"].alias("bytes_removed"),
                    m["executionTimeMs"].alias("exec_ms")).collect())
    print(f"  {label}:")
    for r in rows:
        print(f"    v{r['version']}: inserted {r['ins']}, updated {r['upd']}, copied {r['copied']}, DVs {r['dvs']}, "
              f"files added {r['files']}, bytes added {int(r['bytes_added'] or 0) / 1e6:.1f} MB, "
              f"bytes removed {r['bytes_removed']}, execution {int(r['exec_ms'] or 0) / 1000:.1f} s")


In [0]:
# Stress 14.F.3 — reads: current state, point in time, fact join (Type 2 vs Type 9)
OPEN = "DATE'9999-12-31'"
T9, T2 = TBL_T9_STRESS, TBL_T2_STRESS
TBL_FACT = f"{TBL_T9_STRESS}_fact_sales"
G = q(TYPE3_ATTR)                                                   # group-by attribute
D = q(next((a for a in ATTRS if a != TYPE3_ATTR), TYPE3_ATTR))      # a second attribute (count DISTINCT)

# a synthetic fact table over the wave-1 entities (deterministic, no rand): 3 million sales, Mar-Jun 2025
(spark.range(0, 3_000_000)
 .select(F.col("id").alias("sale_id"),
         wave1_key(F.pmod(F.hash("id", F.lit(7)), F.lit(N_W1_INSERTS))).alias(KEY),
         F.date_add(F.lit(date(2025, 3, 1)), F.pmod(F.hash("id", F.lit(11)), F.lit(122)).cast("int")).alias("sale_date"),
         (F.lit(1) + F.pmod(F.hash("id", F.lit(13)), F.lit(500))).cast("decimal(10,2)").alias("amount"))
 .write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(TBL_FACT))
print(f"fact table: {spark.table(TBL_FACT).count():,} rows")

def bench(sql, runs=3):
    times, res = [], None
    for _ in range(runs):
        t0 = time.perf_counter()
        res = spark.sql(sql).collect()
        times.append(time.perf_counter() - t0)
    return statistics.median(times), [tuple(r) for r in res]

keys = ", ".join(sql_lit(r[0]) for r in spark.sql(
    f"SELECT {K} FROM {T2} WHERE is_current ORDER BY xxhash64({K}) LIMIT 1000").collect())
PIT = "DATE'2025-04-15'"

queries = {
    "current: aggregation": (
        f"SELECT {G}, count(*) n, count(DISTINCT {D}) d FROM {T2} WHERE is_current GROUP BY {G} ORDER BY {G}",
        f"SELECT {G}, count(*) n, count(DISTINCT {D}) d FROM {T9} WHERE element_at(_scd, -1).valid_to = {OPEN} GROUP BY {G} ORDER BY {G}"),
    "current: lookup of 1,000 keys": (
        f"SELECT {K}, {sql_cols(ATTRS[:2])} FROM {T2} WHERE is_current AND {K} IN ({keys}) ORDER BY {K}",
        f"SELECT {K}, {sql_cols(ATTRS[:2])} FROM {T9} WHERE element_at(_scd, -1).valid_to = {OPEN} AND {K} IN ({keys}) ORDER BY {K}"),
    "point in time 2025-04-15: aggregation": (
        f"SELECT {G}, count(*) n FROM {T2} WHERE valid_from <= {PIT} AND valid_to >= {PIT} GROUP BY {G} ORDER BY {G}",
        f"SELECT e.{G}, count(*) n FROM {T9} LATERAL VIEW explode(_scd) x AS e WHERE e.valid_from <= {PIT} AND e.valid_to >= {PIT} GROUP BY e.{G} ORDER BY e.{G}"),
    "fact join: sales by attribute at sale time": (
        f"SELECT d.{G}, count(*) n, sum(f.amount) s FROM {TBL_FACT} f JOIN {T2} d ON f.{K} = d.{K} AND f.sale_date BETWEEN d.valid_from AND d.valid_to GROUP BY d.{G} ORDER BY d.{G}",
        f"SELECT e.{G}, count(*) n, sum(f.amount) s FROM {TBL_FACT} f JOIN {T9} d ON f.{K} = d.{K} LATERAL VIEW explode(d._scd) x AS e WHERE f.sale_date BETWEEN e.valid_from AND e.valid_to GROUP BY e.{G} ORDER BY e.{G}"),
}

# the Part 4 style join: filter + try_element_at on the Type 9 array
q_t9_filter = f"""SELECT v.{G}, count(*) n, sum(amount) s FROM (
  SELECT f.amount, try_element_at(filter(d._scd, e -> e.valid_from <= f.sale_date AND e.valid_to >= f.sale_date), 1) AS v
  FROM {TBL_FACT} f JOIN {T9} d ON f.{K} = d.{K}) WHERE v IS NOT NULL GROUP BY v.{G} ORDER BY v.{G}"""

print(f"{'query':45s} {'Type 2 (s)':>11s} {'Type 9 (s)':>11s}  rows  same result")
for name, (q2, q9) in queries.items():
    s2, r2 = bench(q2)
    s9, r9 = bench(q9)
    print(f"{name:45s} {s2:11.2f} {s9:11.2f}  {len(r2):4d}  {r2 == r9}")
s9f, r9f = bench(q_t9_filter)
r2f = spark.sql(queries["fact join: sales by attribute at sale time"][0]).collect()
print(f"{'fact join, Type 9 filter + try_element_at':45s} {'':>11s} {s9f:11.2f}  {len(r9f):4d}  {[tuple(r) for r in r2f] == r9f}")

In [0]:
# Stress 14.F.4 — how big can the _scd array get, and a late-arriving change
# Synthetic tables that reuse the dimension's own key / attribute NAMES (values are synthetic)
OPEN = "DATE'9999-12-31'"
N_ENT = 100                     # entities per test table
SIZES = [10, 100, 1000, 10000]  # versions per entity
TBL_P = f"{TBL_T9_STRESS}_arr"
PK, CAT, CAT_NAME = q(KEY), q(TYPE3_ATTR), TYPE3_ATTR

def med(sql, runs=3):
    ts = []
    for _ in range(runs):
        t0 = time.perf_counter(); spark.sql(sql).collect(); ts.append(time.perf_counter() - t0)
    return statistics.median(ts)

def element(version, valid_from, valid_to, value):
    return (f"named_struct('version', {version}, 'valid_from', {valid_from}, "
            f"'valid_to', {valid_to}, '{CAT_NAME}', {value})")

def build(n_v):
    t9, t2 = f"{TBL_P}_t9_{n_v}", f"{TBL_P}_t2_{n_v}"
    base = f"""(SELECT i AS {PK}, v,
               date_add(DATE'1990-01-01', v * 30) AS valid_from,
               IF(v = {n_v - 1}, {OPEN}, date_add(DATE'1990-01-01', v * 30 + 29)) AS valid_to,
               concat('cat_', CAST(pmod(i + v, 10) AS STRING)) AS {CAT}
        FROM (SELECT CAST(id AS BIGINT) AS i FROM range({N_ENT}))
        LATERAL VIEW explode(sequence(0, {n_v - 1})) s AS v)"""
    spark.sql(f"""CREATE OR REPLACE TABLE {t2} AS
        SELECT {PK}, {CAT}, valid_from, valid_to, valid_to = {OPEN} AS is_current FROM {base}""")
    spark.sql(f"""CREATE OR REPLACE TABLE {t9} AS
        SELECT {PK}, max_by({CAT}, v) AS {CAT},
               array_sort(collect_list({element('CAST(v AS BIGINT)', 'valid_from', 'valid_to', CAT)})) AS _scd
        FROM {base} GROUP BY {PK}""")
    return t9, t2

spark.sql(f"""CREATE OR REPLACE TEMP VIEW arr_src AS
    SELECT CAST(id AS BIGINT) AS {PK}, 'new_cat' AS {CAT}, 'update' AS change_type,
           DATE'2999-01-01' AS change_date
    FROM range(100)""")

print(f"{N_ENT} entities per table, one MERGE that appends a version to every entity\n")
print(f"{'versions/entity':>15s} | {'Type 9: MERGE s':>15s} {'current s':>9s} {'PIT s':>6s} {'MB':>6s} | "
      f"{'Type 2: MERGE s':>15s} {'current s':>9s} {'PIT s':>6s} {'MB':>6s}")
for n_v in SIZES:
    t9, t2 = build(n_v)
    pit = f"date_add(DATE'1990-01-01', {(n_v // 2) * 30 + 5})"

    t0 = time.perf_counter()
    spark.sql(f"""MERGE INTO {t9} AS tgt USING arr_src AS src ON tgt.{PK} = src.{PK}
        WHEN MATCHED THEN UPDATE SET
          tgt.{CAT} = src.{CAT},
          tgt._scd = concat(
            transform(tgt._scd, e -> IF(e.version = size(tgt._scd) - 1 AND e.valid_to = {OPEN},
                      {element('e.version', 'e.valid_from', 'date_sub(src.change_date, 1)', f'e.{CAT}')}, e)),
            array({element('CAST(size(tgt._scd) AS BIGINT)', 'src.change_date', OPEN, f'src.{CAT}')}))""")
    m9 = time.perf_counter() - t0

    t0 = time.perf_counter()
    spark.sql(f"""MERGE INTO {t2} AS tgt USING (
          SELECT {PK} AS merge_key, * FROM arr_src
          UNION ALL
          SELECT CAST(NULL AS BIGINT), s.* FROM arr_src s JOIN {t2} t ON t.{PK} = s.{PK} AND t.is_current) AS src
        ON tgt.{PK} = src.merge_key AND tgt.is_current
        WHEN MATCHED THEN UPDATE SET tgt.is_current = false, tgt.valid_to = date_sub(src.change_date, 1)
        WHEN NOT MATCHED THEN INSERT ({PK}, {CAT}, valid_from, valid_to, is_current)
          VALUES (src.{PK}, src.{CAT}, src.change_date, {OPEN}, true)""")
    m2 = time.perf_counter() - t0

    c9 = med(f"SELECT {CAT}, count(*) FROM {t9} WHERE element_at(_scd, -1).valid_to = {OPEN} GROUP BY {CAT}")
    c2 = med(f"SELECT {CAT}, count(*) FROM {t2} WHERE is_current GROUP BY {CAT}")
    p9 = med(f"SELECT e.{CAT}, count(*) FROM {t9} LATERAL VIEW explode(_scd) x AS e "
             f"WHERE e.valid_from <= {pit} AND e.valid_to >= {pit} GROUP BY e.{CAT}")
    p2 = med(f"SELECT {CAT}, count(*) FROM {t2} WHERE valid_from <= {pit} AND valid_to >= {pit} GROUP BY {CAT}")
    mb9 = spark.sql(f"DESCRIBE DETAIL {t9}").collect()[0]["sizeInBytes"] / 1e6
    mb2 = spark.sql(f"DESCRIBE DETAIL {t2}").collect()[0]["sizeInBytes"] / 1e6
    print(f"{n_v:15,d} | {m9:15.1f} {c9:9.2f} {p9:6.2f} {mb9:6.1f} | {m2:15.1f} {c2:9.2f} {p2:6.2f} {mb2:6.1f}")
    if n_v != 10:
        spark.sql(f"DROP TABLE IF EXISTS {t9}"); spark.sql(f"DROP TABLE IF EXISTS {t2}")

# --- late-arriving change: a correction dated INSIDE an existing history (entity 1, between two versions' dates)
t9, t2 = f"{TBL_P}_t9_10", f"{TBL_P}_t2_10"
LATE_D = "date_add(DATE'1990-01-01', 135)"   # falls inside version 4 (days 120-149)
cover = f"{PK} = 1 AND valid_from < {LATE_D} AND valid_to >= {LATE_D}"

t0 = time.perf_counter()
spark.sql(f"""UPDATE {t9} SET _scd = transform(array_sort(concat(
      transform(_scd, e -> IF(e.valid_from < {LATE_D} AND e.valid_to >= {LATE_D},
                {element('e.version', 'e.valid_from', f'date_sub({LATE_D}, 1)', f'e.{CAT}')}, e)),
      transform(filter(_scd, e -> e.valid_from < {LATE_D} AND e.valid_to >= {LATE_D}),
                e -> {element('CAST(0 AS BIGINT)', LATE_D, 'e.valid_to', "'late_cat'")})),
      (a, b) -> IF(a.valid_from < b.valid_from, -1, IF(a.valid_from > b.valid_from, 1, 0))),
    (e, i) -> {element('CAST(i AS BIGINT)', 'e.valid_from', 'e.valid_to', f'e.{CAT}')})
  WHERE {PK} = 1""")
l9 = time.perf_counter() - t0

t0 = time.perf_counter()
spark.sql(f"INSERT INTO {t2} SELECT {PK}, 'late_cat', {LATE_D}, valid_to, is_current FROM {t2} WHERE {cover}")
spark.sql(f"UPDATE {t2} SET valid_to = date_sub({LATE_D}, 1), is_current = false WHERE {cover}")
l2 = time.perf_counter() - t0

a = [tuple(r) for r in spark.sql(f"SELECT e.{CAT}, e.valid_from, e.valid_to FROM {t9} LATERAL VIEW explode(_scd) x AS e WHERE {PK} = 1 ORDER BY e.valid_from").collect()]
b = [tuple(r) for r in spark.sql(f"SELECT {CAT}, valid_from, valid_to FROM {t2} WHERE {PK} = 1 ORDER BY valid_from").collect()]
print(f"\nLate-arriving change inside version 4 of entity 1: Type 9 {l9:.1f} s (1 statement), Type 2 {l2:.1f} s (2 statements)")
print(f"history of entity 1 now has {len(a)} versions in Type 9 and {len(b)} rows in Type 2; identical: {a == b}")
for r in a[3:7]:
    print("  ", tuple(str(x) for x in r))
spark.sql(f"DROP TABLE IF EXISTS {t9}"); spark.sql(f"DROP TABLE IF EXISTS {t2}")

### 14.F Findings - Type 2 baseline vs Type 9 on the same random feed

- **Same history, verified.** Expanding `_scd` gives exactly the rows the Type 2 baseline holds: 1,090,384 rows, 0 differences (`exceptAll` both ways).
- **Load.** Type 2 baseline 15.3 s vs Type 9 16.4 s for the same three waves (run of 2026-11-18; 14.B and 14.F.1 print the figures of the latest run), one `ALTER` (new column) not included - a tie for practical purposes.
- **Disk.** Type 2 baseline 7.7 MB vs Type 9 ≈ 11–12 MB (12.2 MB in the 2026-11-18 run; 14.F.2 prints the latest). This is where Type 9 loses: nested arrays compress worse than flat rows, and the current version is stored twice (scalar columns + last `_scd` element).
- **Reads.** Current-state aggregation, a 1,000-key lookup and a point-in-time aggregation are on par. In the fact join, `explode` is on par with Type 2, while the `filter + try_element_at` variant is slower (1.6 s vs 1.0 s in the 2026-11-18 run).
- **Array size.** Arrays of up to 10,000 versions per entity were created and queried without trouble.
- **Late-arriving change inside an old version.** Type 9: one statement. Type 2: two statements (split the row, then close the previous one).
- **Revival rule, one for the whole notebook.** A deleted entity is never re-opened: the guard in section 12, the Type 9 `MERGE` in 14.B and the Type 2 baseline in 14.F.1 all skip events for entities whose last version is closed (8 update events in this feed; 14.B prints them). Earlier runs re-opened the 4 deleted demo products in both tables.
- **Schema.** All `_scd` elements share one struct type: a field added later reads as NULL in older elements; a single version cannot carry its own set of fields.
- **14.G.0 / 14.G.** Runtime, table properties, raw `_scd` of products 1, 8 and 30, the join plan and a `VACUUM` test on a `DEEP CLONE`: after a short retention and `VACUUM`, time travel to old Delta versions is refused, while the embedded `_scd` history stays intact.


In [0]:
# 14.G.0 - VACUUM safety test on a DEEP CLONE (never on the original): short retention set as a table property
CL = f"{TBL_T9_EVOLUTION}_vacuum_clone"
K0 = sql_lit(EVO_KEYS[0])
spark.sql(f"CREATE OR REPLACE TABLE {CL} DEEP CLONE {TBL_T9_EVOLUTION}")
spark.sql(f"UPDATE {CL} SET {q(ATTRS[0])} = {q(ATTRS[0])} WHERE {K} = {K0}")
try:
    spark.sql(f"OPTIMIZE {CL}")
except Exception as ex:
    print("OPTIMIZE skipped:", str(ex)[:150])
spark.sql(f"ALTER TABLE {CL} SET TBLPROPERTIES ('delta.deletedFileRetentionDuration' = 'interval 0 seconds', "
          f"'delta.logRetentionDuration' = 'interval 0 seconds')")
time.sleep(5)
print("clone history:", [(h["version"], h["operation"]) for h in
                         spark.sql(f"DESCRIBE HISTORY {CL}").select("version", "operation").collect()])
try:
    vr = spark.sql(f"VACUUM {CL}").collect()
    print("VACUUM (table-property retention) result:", vr[0][0] if vr else "ok")
except Exception as ex:
    print("VACUUM failed:", str(ex)[:300])
for v in (0, 1):
    try:
        n = spark.sql(f"SELECT count(*) FROM {CL} VERSION AS OF {v}").collect()[0][0]
        print(f"time travel VERSION AS OF {v} after VACUUM: rows {n}")
    except Exception as ex:
        print(f"time travel VERSION AS OF {v} after VACUUM FAILED:", str(ex)[:160])
print("current rows:", spark.table(CL).count(),
      f"| embedded versions of {KEY} = {EVO_KEYS[0]}:",
      spark.sql(f"SELECT size(_scd) FROM {CL} WHERE {K} = {K0}").collect()[0][0])
spark.sql(f"DROP TABLE IF EXISTS {CL}")

In [0]:
# 14.G - Extra checks quoted in the articles: runtime, table properties, raw _scd, query plan (VACUUM test: see 14.G.0)
print("Runtime (spark.version):", spark.version)

print("\nSHOW TBLPROPERTIES on the Type 9 evolution table:")
for r in spark.sql(f"SHOW TBLPROPERTIES {TBL_T9_EVOLUTION}").collect():
    print("  ", r[0], "=", r[1])
d = spark.sql(f"DESCRIBE DETAIL {TBL_T9_EVOLUTION}").collect()[0]
print("DESCRIBE DETAIL: numFiles", d["numFiles"], "| sizeInBytes", d["sizeInBytes"])

sample_keys = EVO_KEYS[:3]
print(f"\nSchema-evolution rows: raw _scd of {KEY} in {sample_keys}")
for r in spark.sql(f"SELECT {K}, to_json(_scd) AS j FROM {TBL_T9_EVOLUTION} "
                   f"WHERE {K} IN ({', '.join(sql_lit(k) for k in sample_keys)}) ORDER BY {K}").collect():
    print(r[KEY], r["j"])

print("\nEXPLAIN FORMATTED - Type 9 fact join (filter + try_element_at):")
fact = spark.table(TBL_FACT)
dcol = [c for c in fact.columns if "date" in c][0]
print("fact columns:", fact.columns)
try:
    plan_sql = (f"SELECT count(*) AS n FROM {TBL_FACT} f JOIN {TBL_T9_STRESS} d ON d.{K} = f.{K} "
                f"WHERE try_element_at(filter(d._scd, e -> f.{dcol} BETWEEN e.valid_from AND e.valid_to), 1).{q(TYPE3_ATTR)} IS NOT NULL")
    print(spark.sql("EXPLAIN FORMATTED " + plan_sql).collect()[0][0][:2500])
except Exception as ex:
    print("EXPLAIN failed:", str(ex)[:300])

# The VACUUM safety test on a DEEP CLONE lives in 14.G.0 (table-property retention),
# which works on serverless; it is not repeated here.

In [0]:
# 14.G.2 - Scan plans quoted in Part 4: what is read for the current-state query and for the exploded history
def scan_lines(sql):
    txt = spark.sql("EXPLAIN FORMATTED " + sql).collect()[0][0]
    return [l.strip() for l in txt.split("\n") if re.search(r"ReadSchema|PushedFilters|Format:|Output \[", l)]

q_cur = f"SELECT {sql_cols(cols())} FROM {TBL_T9_EVOLUTION}"
q_his = (f"SELECT {K}, v.version, v.valid_from, v.valid_to, v.{q(TYPE3_ATTR)} "
         f"FROM {TBL_T9_EVOLUTION} LATERAL VIEW explode(_scd) x AS v")
for name, stmt in [("current state (columns only, no _scd)", q_cur), ("exploded history", q_his)]:
    print("==", name)
    for l in scan_lines(stmt):
        print("  ", l[:600])


# 15. Summary — the SCD types 0–8 revisited, and Type 9 (the `_scd` array) evaluated

## 15.A What the notebook demonstrated about the classic types (0–8)

| Type | Author | Where history lives | Key finding in this notebook | Main cost |
|---|--------|---|---|---|
| **0 – fixed** | Ralph Kimball / Kimball Group | Nowhere (by design) | Insert-only MERGE: updates and deletes in the feed are simply ignored; the first value survives forever | Cannot answer *any* as-of question |
| **1 – overwrite** | Ralph Kimball | Nowhere; the row is the entity | Every MERGE collapses to the latest values; the Delta log is the only record that anything changed | Complete amnesia once retention/VACUUM passes |
| **2 – row versioning** | Ralph Kimball | Extra rows (one per change) | `valid_from` / `valid_to` / `is_current` give correct as-of joins; the demo showed current + history views split naturally | Table grows with every attribute change; all queries must filter one row per key |
| **3 – in-place history columns** | Ralph Kimball | Columns on the same row (`previous_*`) | Keeps one row per entity (no fan-out) but stores only ONE step back — a second change overwrites the earlier "previous" value | Truncated history; schema must be pre-designed |
| **4 – mini-dimension** | Ralph Kimball | Volatile attributes split into a separate mini-dimension (one row per attribute combination); the base row carries only the current `mini_sk` | The mini-dim absorbs the churn the base table never sees; the MERGE re-points `mini_sk` Type 1-style; history is reconstructible only through the facts' `mini_sk` references | Two tables + surrogate-key joins on every fact; no point-in-time view from the dimension itself |
| **5 – mini-dim + Type 1 outrigger** | Ralph Kimball | Type 4 mini-dim + the current group's attributes copied into the base dimension (`current_category`, `current_brand`) | Dashboards filter on the current group straight from the base dimension; the outrigger is overwritten in the same MERGE that re-points `mini_sk` | Everything Type 4 costs, plus outrigger columns to keep in sync |
| **6 – hybrid (Type 2 rows + Type 1 current column)** | Ralph Kimball | Extra rows (Type 2) + a current-value column overwritten on every history row | Facts group either by the value in effect at the time (`category`) or by today's value (`current_category`) | A second MERGE re-stamps `current_category` across every history row on each change |
| **7 – dual (1 + 2 tables)** | Kimball Group | Regular snapshot table + full history table | Simple "current" queries stay fast, history is complete in the second table | History is a separate object to build, sync and govern |
| **8 – fact-date rows (Ivanov 2017)** | Oleg Ivanov | Rows aligned to the facts (one per entity × fact date + special dates 1900-01-01 / 9999-12-31) | Attributes frozen at every fact date; the equality join `sale_date = event_date` replaces the `BETWEEN` range scan and prunes to one `event_date` partition | Rows multiply with the facts (6× storage in the article's example); restatements and deletes need extra handling |
| **9 – embedded history (concept)** | Bartosz Malec | Inside the row (`_scd` array) | One row per entity carries the complete timeline; current columns serve "now" without touching `_scd` | Whole-array rewrite per update; hand-rolled merge logic |

Cross-cutting findings:

* **None of types 0–8 stores history inside the row itself** — it lives in extra rows, extra columns, or extra tables. Type 9 is the only type where one row carries the entity's complete 
business timeline as an `_scd` ARRAY<STRUCT>.
* **All types depend on the Delta log** for *system-time* questions (when a row was written). Chapter 13 showed the two time axes (business `valid_from`/`valid_to` vs. commit timestamps / 
Change Data Feed) and how they reconcile: the schema-evolution feed grew `sum(size(_scd))` by exactly the number of events between the Delta version before and after the schema-evolution MERGE.
* **Schema evolution** (chapter 12): the Type 9 feed introduced `supplier` and `warranty_months` mid-stream. A metadata-only `ALTER TABLE ... ADD COLUMNS` (including `_scd.
element.*`) widened the struct in place — older versions read the new fields as NULL ("not known at that time"), newer versions carry values. No backfill, no flag columns, no table rewrite.
* **Deletes without a flag column**: a deleted entity is recognized because the LAST `_scd` version's `valid_to` is no longer `9999-12-31` — history is preserved instead of destroyed.

## 15.B Type 8 (Oleg Ivanov, 2017) — what the notebook demonstrated

Built strictly per the article's criteria: a single `event_date` (no `valid_from`/`valid_to`, no flag), one row per `product_id` × sale date, special dates `1900-01-01` (initial state, SCD 0) and `9999-12-31` (current state, SCD 1), and a Delta table **partitioned by `event_date`**.

**The article's trade, measured on the product sample (section 6):**

* the same data needs **61 SCD 2 version rows**, but **191 SCD 8 rows** = 22 initial (`1900-01-01`) + 151 fact-date + 18 current (`9999-12-31`) → **3.1× more rows** — disk space traded for read speed, exactly as the article predicts (its example: 6× space for a 21× smaller read);
* the September partition holds **21 of the 191 rows** — a month-bounded query reads **9.1× fewer rows** than the whole table, via partition pruning alone (`SHOW PARTITIONS` confirms 15 partitions, one per date incl. the two special dates);
* the as-of join is a plain **equality** `sale_date = event_date` — no `BETWEEN`, no surrogate key, no `is_current` filter.

**MERGE behaviour (section 8):** new fact dates insert via `WHEN NOT MATCHED`; a data-quality correction for product 3's 2024-10-01 row (valid for that date only) was applied via `WHEN MATCHED`; a late brand change effective 2024-12-20 overwrote the `9999-12-31` row (SCD 1 bonus) **and** the fact-date rows dated on or after 2024-12-20, while earlier fact-date rows kept the value that was true at their date.

**Beyond the article:** deletes are handled explicitly (deleted products stop producing fact dates and get no `9999-12-31` row) and restatements are shown via MERGE — the two cases Oleg Ivanov's article leaves open.

## 15.C Type 9 (`_scd` array) — pros and cons

**Pros**

1. **Complete history in one row** — the full version timeline (values, `valid_from`, `valid_to`, `version`) travels with the entity; no fan-out, no second table, no duplicated keys in the 
current snapshot.
2. **Native as-of queries** — `filter(_scd, e -> e.valid_from <= d AND e.valid_to >= d)` answers "what was true on date d" with a single array predicate.
3. **Elastic schema** — the struct grows by adding fields (top-level columns *and* `_scd.element.*`); old versions legitimately hold NULLs for fields that did not exist yet, which is the 
honest representation.
4. **Metadata-only evolution** — Delta's `ALTER TABLE ADD COLUMNS` rewrites no data files; the chapter-12 history records the evolution as ordinary commits.
5. **Compact and Delta-friendly incremental loads** — the MERGE touches only rows whose `_scd` grew (`size(s._scd) <> size(t._scd)`), and Change Data Feed exposes exactly which entities 
evolved.
6. **Survives VACUUM and log retention** — business history is data, not log; it outlives the log-retention and VACUUM windows that bound time travel (Delta defaults: delta.logRetentionDuration = 30 days, delta.deletedFileRetentionDuration = 7 days; 13.D shows the values set on this table).
7. **No `is_current` flag to forget** — currency is derived (`element_at(_scd, size(_scd)).valid_to = DATE'9999-12-31'`), so there is no is_current flag that can drift out of sync with the dates (closed/deleted entities still sit in the table and are recognised by the last valid_to, so a plain read of the current columns also returns them unless you filter).

**Cons**

1. **Array-of-struct cost** — every row carries its whole history; entities with hundreds of versions bloat rows and degrade column pruning on `_scd`.
2. **No engine-level SCD helper** — the merge logic (close previous version, number versions, rebuild the array) is hand-written PySpark/SQL; there is no `apply_changes`-style automation for 
Type 9.
3. **Rewrites the array on every change** — a one-field update rewrites the entire `_scd` array of that row (arrays grow by whole new elements); write amplification is worse than Type 1's 
in-place update.
4. **Struct-wide schema rigidity** — all array elements share ONE struct type; you cannot drop or retype a field without rewriting, and every version element carries every field.
5. **Awkward analytics** — tools that cannot `explode` arrays need a view layer; fact joins against a specific version always require the explode/filter pattern, which is more work than a 
plain key join.
6. **No per-attribute granularity** — elements store full row images per version, not per-attribute changes; a Type 4-style split of volatile vs. stable attributes is not possible inside one 
`_scd` element.
7. **System-time still external** — Type 9 records *business* time only; "when was this version loaded" still requires the Delta log / CDF (chapter 13), so the bi-temporal picture always 
spans two systems.

## 15.D When to use Type 9 — use cases

**Strong fits**

* **Reference/master data with moderate change frequency** (products, customers, accounts) where the full timeline must be queryable from the current table — a few dozen versions per entity, 
not thousands.
* **As-of reporting in place** — "what was this entity's state on 2024-09-15?" answered without a second table or time travel.
* **Frequently evolving source schemas** — feeds that switch on new attributes mid-stream; the struct widens with NULL-padded history instead of triggering backfills or table rewrites.
* **Retention beyond the Delta log** — compliance or audit needs where business history must survive `VACUUM` and log expiry.
* **Portable history** — one table export carries the whole timeline to another environment; no dependency on a source system's history table.
* **Teaching and small dimensions** — where single-object simplicity outweighs write amplification.

**Weak fits (prefer another type)**

* **High-churn, volatile attributes** (prices, balances, click-stream state) — Type 5's mini-dim or a fact-with-versions model keeps the dimension row stable.
* **Very large dimensions with heavy version counts** — Type 2 rows (or Type 6 + history table) prune and compress better than giant arrays.
* **Workloads where BI users join facts by plain surrogate key** — classic Type 2 with `is_current` filtering is simpler for them than explode-and-filter on every query.
* **Attributes that must evolve independently per field** — Type 3-style columns or the Type 4 split offer field-level history the single struct cannot.

**Bottom line**: Type 9 trades write amplification and hand-rolled merge logic for *self-contained, schema-elastic, retention-proof history inside the row*. Paired with the Delta log (system 
time) it gives a bi-temporal dimension in a single table — the pattern of choice when the timeline must outlive the log and travel with the data, and the wrong one when version volume or BI 
join simplicity dominates.

### 16. Clean-up — drop the schema (commented out)

Uncomment the block below to drop the entire `{catalog}.{schema}` schema with `CASCADE`, removing every table this notebook created during its run (MERGE examples for types 0–9, the schema-evolution table, and the stress-test table) in one statement. It builds `SCHEMA` from `catalog_name` and `schema_name` of the parameters cell (section 0), so it always drops exactly the schema configured there.


In [0]:
# SCHEMA = f"{catalog_name}.{schema_name}"   # from the parameters cell (section 0)
# spark.sql(f"DROP SCHEMA IF EXISTS {SCHEMA} CASCADE")
# print("dropped", SCHEMA)